# EDA LFP — Pipeline v9

Pipeline completo de análise exploratória de sinais LFP com features avançadas.

> **v10:** adicionada a Fase 10B (centroide espectral), que resume o espectro de cada
> segundo em uma única frequência contínua (Hz) por canal — complementar às potências
> por banda da Fase 10A. As fases 10B em diante foram renumeradas.

> **v12:** adicionada a etapa pós-pipeline de **Tendência**. Em vez de salvar
> colunas `_zscore` (por segundo) em `features_all`, calcula-se o slope
> (`_tendencia`) do z-score por trial ao longo do tempo — 1 valor por trial
> em vez de 1 por segundo. Trials com `nor` no nome (`stem`) não são
> processados nesta etapa.

> **v11:** adicionada a Fase 10D. A área anatômica de cada canal agora é lida
> dinamicamente de `planilha_ratos_R1_R7.xlsx` (qualquer rato novo adicionado na
> planilha, ex. R8, é reconhecido automaticamente). As análises de Welch/PSD,
> harmônicas, espectrograma, centroide e PAC passam a ser feitas **por área**
> (média dos canais bons daquela área) em vez de por canal. As janelas individuais
> do Welch (segmentos brutos + PSD por janela) são salvas em `welch_janelamento/`.

| Fase | Conteúdo |
|------|----------|
| **0** | Imports, configurações globais e parâmetros |
| **1** | Leitura dos arquivos `.int` + anotações do CSV |
| **2** | Exclusão do rato R8 |
| **3** | Identificação de canais ruins (automática + anotações CSV) |
| **4** | Inspeção visual do sinal bruto completo por canal |
| **5** | Remoção de ruído 55–65 Hz (notch) |
| **6** | Remoção de DC offset |
| **7** | Downsample + salvamento `sinal_downsample` |
| **8** | Conversão para arquivos de sinal por segundo |
| **9** | Features de estatísticas por canal |
| **10A** | PSD Welch (Hanning, 1s, 50% overlap, NFFT=1024) + bandas |
| **10B** | Centroide espectral — frequência contínua por canal |
| **10D** | Sinal médio por área anatômica + janelamento do Welch (`welch_janelamento/`) — a partir daqui a análise é **por área**, não por canal |
| **10C** | Detecção do pico Theta real + harmônicas (controle PAC) |
| **11** | Wavelet de Morlet |
| **12** | CFC — Phase-Amplitude Coupling (PAC) |
| **13** | Z-score de potências por banda por canal |
| **14** | Espectrograma multi-painel por banda |
| **15** | Figura 1 — Histologia e registros eletrofisiológicos |
| **Pipeline** | Execução completa por arquivo (seletor `RATO_PIPELINE`) |
| **Pós-pipeline** | Tendência (slope) por trial (substitui _zscore por _tendencia em `features_all`, ignora sessões `NOR`) | Z-score comparativo entre seções | PAC | Sumário de outputs |

---

> **Como rodar um rato específico:** na célula Pipeline, altere `RATO_PIPELINE = 'R1'`  
> **Como rodar todos:** `RATO_PIPELINE = 'ALL'`


---
## Fase 0 — Imports e Configurações Globais

In [3]:
import re
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

from scipy.signal import (
    welch, spectrogram, iirnotch, filtfilt,
    resample_poly, butter, sosfiltfilt
)
from scipy.fft import rfft, rfftfreq
from scipy.stats import kurtosis, skew, zscore
import pywt  # pip install PyWavelets

warnings.filterwarnings('ignore')
matplotlib.use('Agg')

# ══════════════════════════════════════════════════════════════
# PARÂMETROS GLOBAIS
# ══════════════════════════════════════════════════════════════
FS_ORIGINAL = 25_000   # Hz — frequência de aquisição do Intan
FS          = 1_000    # Hz — frequência alvo após downsample
DS_FACTOR   = FS_ORIGINAL // FS  # fator de decimação = 20
N_CANAIS    = 16

# Notch 55–65 Hz
NOTCH_LO = 55.0
NOTCH_HI = 65.0
NOTCH_CENTER = (NOTCH_LO + NOTCH_HI) / 2.0  # 60 Hz
NOTCH_Q = NOTCH_CENTER / (NOTCH_HI - NOTCH_LO)  # Q = 6

# Welch
WELCH_NFFT    = 1024
WELCH_NPERSEG = FS * 1          # 1 s
WELCH_NOVERLAP = WELCH_NPERSEG // 2  # 50%
WELCH_WINDOW  = 'hann'

# Bandas espectrais de INTERESSE FISIOLÓGICO (Hz)
# As harmônicas de theta NÃO são bandas fixas aqui.
# Elas são calculadas dinamicamente a partir do pico theta real
# (ver Fase 10C) e usadas como CONTROLE METODOLÓGICO do PAC,
# não como variáveis dependentes.
BANDAS = {
    'delta':        (1,   4),
    'theta':        (6,  12),
    'beta':         (13, 30),
    'gamma_lento':  (25, 55),
    'gamma_rapido': (65, 110),
}

# Janela de tolerância (±Hz) para considerar que uma harmônica
# cai dentro de uma banda de gamma (controle de PAC)
HARMONICA_TOLERANCIA_HZ = 3.0

# Cores para plots
CORES_BANDAS = {
    'delta':        '#9b59b6',
    'theta':        '#3498db',
    'beta':         '#e67e22',
    'gamma_lento':  '#2ecc71',
    'gamma_rapido': '#e74c3c',
}

# Caminhos (ajuste conforme sua estrutura de pastas)
BASE_DIR    = Path(r'..\Experimento_eletro')   # raiz do experimento
OUTPUT_DIR  = Path('EDA_outputs')
OUTPUT_DIR.mkdir(exist_ok=True)

# Rato a excluir
RATO_EXCLUIR = ''

# ── Mapeamento canal → área anatômica (lido dinamicamente da planilha) ────────
# Fonte: planilha_ratos_R1_R7.xlsx (aba "Ratos"). Uma linha por rato (ex: R1..R8),
# colunas "Canal 0".."Canal 15" com o nome da área (ex: "DGV molecular 1").
# O sufixo numérico (" 1"/" 2") é removido para agrupar os dois canais da mesma
# área anatômica sob o mesmo nome (ex: "DGV molecular").
PLANILHA_AREAS_PATH = Path(r'..\Experimento_eletro\planilha_ratos_R1_R7.xlsx')


def carregar_canal_areas(caminho_xlsx=None, n_canais=N_CANAIS):
    """Lê a planilha de áreas anatômicas e monta {RATO: [area_ch0, ..., area_ch15]}.

    Lê TODAS as linhas presentes na planilha — se um novo rato (ex: R8) for
    adicionado como uma nova linha, ele passa a existir automaticamente aqui,
    sem precisar editar código.
    """
    caminho_xlsx = Path(caminho_xlsx) if caminho_xlsx else PLANILHA_AREAS_PATH
    df_areas = pd.read_excel(caminho_xlsx, sheet_name=0)
    df_areas.columns = [str(c).strip() for c in df_areas.columns]

    col_rato = df_areas.columns[0]
    canal_cols = [c for c in df_areas.columns if c.lower().startswith('canal')]
    canal_cols = sorted(canal_cols, key=lambda c: int(re.search(r'\d+', c).group()))

    mapa = {}
    for _, row in df_areas.iterrows():
        rato = str(row[col_rato]).strip().upper()
        if not rato or rato == 'NAN':
            continue
        areas = []
        for c in canal_cols:
            val = row[c]
            area = '' if pd.isna(val) else str(val).strip()
            area = re.sub(r'\s+\d+$', '', area)  # remove sufixo ' 1' / ' 2' → agrupa a área
            areas.append(area)
        while len(areas) < n_canais:
            areas.append('')
        mapa[rato] = areas[:n_canais]
    return mapa


try:
    CANAL_AREAS = carregar_canal_areas()
except Exception as e:
    print(f'⚠️  Não foi possível carregar CANAL_AREAS de {PLANILHA_AREAS_PATH}: {e}')
    print('   Ajuste PLANILHA_AREAS_PATH e rode esta célula novamente.')
    CANAL_AREAS = {}


def get_area_canal(rato, canal_idx):
    """Retorna a área anatômica do canal (0-based) para o rato informado."""
    areas = CANAL_AREAS.get(str(rato).upper(), [])
    if canal_idx < len(areas) and areas[canal_idx]:
        return areas[canal_idx]
    return f'ch{canal_idx+1:02d}'



print(f'✅ Configurações carregadas')
print(f'   FS_ORIGINAL={FS_ORIGINAL} Hz → FS_DOWN={FS} Hz (fator {DS_FACTOR}x)')
print(f'   Notch: {NOTCH_LO}–{NOTCH_HI} Hz (centro {NOTCH_CENTER} Hz, Q={NOTCH_Q:.1f})')
print(f'   Welch: nperseg={WELCH_NPERSEG} (1 s) | noverlap={WELCH_NOVERLAP} | nfft={WELCH_NFFT} | janela={WELCH_WINDOW}')
print(f'   Bandas: {list(BANDAS.keys())}')
print(f'   Excluir: {RATO_EXCLUIR}')

✅ Configurações carregadas
   FS_ORIGINAL=25000 Hz → FS_DOWN=1000 Hz (fator 25x)
   Notch: 55.0–65.0 Hz (centro 60.0 Hz, Q=6.0)
   Welch: nperseg=1000 (1 s) | noverlap=500 | nfft=1024 | janela=hann
   Bandas: ['delta', 'theta', 'beta', 'gamma_lento', 'gamma_rapido']
   Excluir: 


---
## Fase 1 — Leitura dos arquivos `.int` + anotações do CSV

**O que faz:**  
- Varre recursivamente `BASE_DIR` em busca de arquivos `.int`  
- Carrega o Excel `dados_sessões_comportamento_acontecimentos.xlsx` com anotações de cada sessão  
- Faz o join entre o nome do arquivo e as anotações (rato, condição, sessão, acontecimento)  

**Por que:**  
Centralizar a leitura aqui garante rastreabilidade — cada arquivo `.int` carrega seus metadados comportamentais desde o início.

In [4]:
from pathlib import Path
import sys

BASE_DIR = Path.cwd().parent
sys.path.insert(0, str(BASE_DIR))

from Downsample.read_intan_data import read_intan_data # função padrão do lab

# ── 1.1 Carregar anotações do Excel ──────────────────────────────────────────
XLSX_PATH = BASE_DIR / 'Experimento_eletro' / 'dados_sessões_comportamento_acontecimentos.xlsx'

dfs_anotacoes = []
xl = pd.ExcelFile(XLSX_PATH)
for sheet in xl.sheet_names:
    df_s = xl.parse(sheet)
    # Normaliza nome das colunas
    df_s.columns = [str(c).strip() for c in df_s.columns]
    # Mantém apenas linhas com Rato preenchido
    if 'Rato' in df_s.columns:
        df_s = df_s[df_s['Rato'].notna()]
        dfs_anotacoes.append(df_s)

df_anot = pd.concat(dfs_anotacoes, ignore_index=True)

# Normaliza colunas-chave
df_anot['Rato']    = df_anot['Rato'].astype(str).str.strip().str.upper()
df_anot['Condição'] = df_anot['Condição'].astype(str).str.strip()
df_anot['Sessão']  = df_anot['Sessão'].astype(str).str.strip().str.upper()
df_anot['acontencimento'] = df_anot.get('acontencimento', pd.Series(dtype=str)).fillna('')

print(f'✅ Anotações carregadas: {len(df_anot)} linhas, {df_anot["Rato"].nunique()} ratos')
print(df_anot[['Rato','Condição','Sessão','acontencimento']].head(10).to_string(index=False))

✅ Anotações carregadas: 258 linhas, 10 ratos
Rato Condição Sessão                  acontencimento
  R1      NOR     A1                                
  R1      NOR     A2                                
  R1      NOR     A3                                
  R1      NOR     A4                                
  R1      NOR     T1 Objeto caiu / sinal desconectou
  R1      NOR     T2                                
  R1      NOR     T3                                
  R1        s     T4                                
  R1     0.25     A1                                
  R1     0.25     A2                                


In [3]:
# ── 1.2 Varrer arquivos .int ─────────────────────────────────────────────────

def parse_nome_arquivo(stem):
    """Extrai rato, condição/secao e trial a partir do nome do arquivo.
    Exemplos: r1_25_T1_... → R1, 0.25, T1
              R3NORA2_...  → R3, NOR, A2
    """
    stem_up = stem.upper()

    # Padrão novo: r1_25_T1 / r7_50_A2
    m = re.match(r'(R\d+)_(\d+)_([AT]\d+)', stem_up)
    if m:
        rato, sec, trial = m.group(1), m.group(2), m.group(3)
        condicao = str(int(sec) / 100)  # '25' → '0.25'
        return rato, condicao, trial

    # Padrão antigo: R3NORA2 / R4NORT1
    m = re.match(r'(R\d+)(NOR)([AT]\d+)', stem_up)
    if m:
        return m.group(1), 'NOR', m.group(3)

    return None, None, None


arquivos_int = sorted(BASE_DIR.rglob('*.int'))

registros = []
for p in arquivos_int:
    rato, condicao, trial = parse_nome_arquivo(p.stem)
    registros.append({
        'path': p,
        'stem': p.stem,
        'rato': rato,
        'condicao': condicao,
        'trial': trial,
    })

df_arquivos = pd.DataFrame(registros)
print(f'✅ Arquivos .int encontrados: {len(df_arquivos)}')
print(f'   Ratos: {sorted(df_arquivos["rato"].dropna().unique())}')

✅ Arquivos .int encontrados: 274
   Ratos: ['R1', 'R2', 'R3', 'R4', 'R5', 'R7', 'R8']


In [4]:
# ── 1.3 Join com anotações ───────────────────────────────────────────────────
# Normaliza chave de join: rato + condição + trial
df_anot['_key'] = (df_anot['Rato'] + '_' +
                   df_anot['Condição'].astype(str) + '_' +
                   df_anot['Sessão'])

df_arquivos['_key'] = (df_arquivos['rato'].fillna('') + '_' +
                       df_arquivos['condicao'].fillna('') + '_' +
                       df_arquivos['trial'].fillna(''))

df_arquivos = df_arquivos.merge(
    df_anot[['_key', 'acontencimento', 'Sono_pre', 'Sono_pos']].drop_duplicates('_key'),
    on='_key', how='left'
)
df_arquivos['acontencimento'] = df_arquivos['acontencimento'].fillna('')

print('✅ Join realizado. Exemplo de linhas com anotação:')
print(df_arquivos[df_arquivos['acontencimento'] != ''][['stem','rato','condicao','trial','acontencimento']]
      .head(10).to_string(index=False))

✅ Join realizado. Exemplo de linhas com anotação:
                  stem rato condicao trial                    acontencimento
r1_25_T1_240416_124853   R1     0.25    T1                        Sinal ruim
r1_25_T2_240416_130351   R1     0.25    T2                        Sinal ruim
r1_25_T3_240416_132013   R1     0.25    T3 Arquivo extra / falha tentativa 3
r1_25_T3_240416_132016   R1     0.25    T3 Arquivo extra / falha tentativa 3
r1_25_T4_240416_133540   R1     0.25    T4                 Falha tentativa 4
r1_75_T1_240420_122202   R1     0.75    T1                        Sinal ruim
r1_75_T2_240420_123826   R1     0.75    T2                        Sinal ruim
r1_75_T3_240420_125313   R1     0.75    T3       Sinal igual min 32,35,40,46
r1_75_T4_240420_130736   R1     0.75    T4                     Dormiu min 41
r2_25_A2_240501_122518   R2     0.25    A2                   Derrubou objeto


---
## Fase 2 — Exclusão do rato R8

**Por que excluir R8:**  
O rato R8 apresentou problemas sistemáticos de sinal ao longo de múltiplas sessões: ruído no sinal nas sessões NOR (A4, T2, T4), ausência de registro eletrofisiológico (25% A2 — `sem registro eletro`), e sessões de 0.5% com status indefinido (`KD?`). A qualidade do dado é insuficiente para inclusão na análise.

In [5]:
# ── Fase 2 — Exclusão do R8 e Descarte por Falhas no Experimento ──────────────
n_antes = len(df_arquivos)

# 1. Identifica e separa os arquivos do R8 para o log de descarte
df_r8 = df_arquivos[df_arquivos['rato'] == RATO_EXCLUIR].copy()
print(f'🗑️  Arquivos do {RATO_EXCLUIR} a excluir por critério de animal: {len(df_r8)}')
if not df_r8.empty:
    print(df_r8[['stem','condicao','trial','acontencimento']].to_string(index=False))

print('\n' + '─'*60 + '\n')

# 2. Identifica e separa arquivos com 'falha' na coluna acontecimento (ignorando maiúsculas/minúsculas)
mask_falha = df_arquivos['acontencimento'].str.lower().str.contains('falha', na=False)
df_falhas = df_arquivos[mask_falha].copy()
print(f'🗑️  Arquivos descartados por "falha" na anotação comportamental: {len(df_falhas)}')
if not df_falhas.empty:
    print(df_falhas[['stem','condicao','trial','acontencimento']].to_string(index=False))

# 3. Aplica AMBOS os filtros de exclusão no dataframe principal (Garante a saída do R8 e das falhas)
df_arquivos = df_arquivos[
    (df_arquivos['rato'] != RATO_EXCLUIR) & (~mask_falha)
].reset_index(drop=True)

n_removidos_total = n_antes - len(df_arquivos)
print('\n' + '═'*60)
print(f'✅ Limpeza concluída! Arquivos restantes: {len(df_arquivos)} (Total removido: {n_removidos_total})')
print(f'   Ratos válidos no dataset: {sorted(df_arquivos["rato"].dropna().unique())}')

🗑️  Arquivos do  a excluir por critério de animal: 0

────────────────────────────────────────────────────────────

🗑️  Arquivos descartados por "falha" na anotação comportamental: 3
                  stem condicao trial                    acontencimento
r1_25_T3_240416_132013     0.25    T3 Arquivo extra / falha tentativa 3
r1_25_T3_240416_132016     0.25    T3 Arquivo extra / falha tentativa 3
r1_25_T4_240416_133540     0.25    T4                 Falha tentativa 4

════════════════════════════════════════════════════════════
✅ Limpeza concluída! Arquivos restantes: 271 (Total removido: 3)
   Ratos válidos no dataset: ['R1', 'R2', 'R3', 'R4', 'R5', 'R7', 'R8']


---
## Fase 2.5 — Filtro: apenas arquivos de Teste T

**O que faz:**  
Mantém em `df_arquivos` somente os registros cujo trial começa com **`T`** (ex: T1, T2, T3, T4).  
Arquivos de aquisição/habituação (**A1–A4**) são descartados desta análise.  

**Por que:**  
A hipótese central do TCC diz respeito ao momento do **teste de memória** (exposição ao objeto novo).  
Os trials de aquisição servem apenas como habituação e não entram na análise de features nem no pipeline de ML.

In [6]:
# ── Fase 2.5 — Mantém apenas trials de Teste (T1, T2, T3, T4) ─────────────
n_antes_filtro = len(df_arquivos)

df_arquivos = df_arquivos[
    df_arquivos['trial'].fillna('').str.upper().str.startswith('T')
].reset_index(drop=True)

n_descartados = n_antes_filtro - len(df_arquivos)

print(f'✅ Filtro Teste T aplicado')
print(f'   Antes : {n_antes_filtro} arquivos')
print(f'   Após  : {len(df_arquivos)} arquivos  (descartados {n_descartados} trials de aquisição A)')
print(f'   Ratos : {sorted(df_arquivos["rato"].dropna().unique())}')
print()
print('Distribuição por condição e trial:')
print(df_arquivos.groupby(['condicao','trial']).size().to_string())

✅ Filtro Teste T aplicado
   Antes : 271 arquivos
   Após  : 103 arquivos  (descartados 168 trials de aquisição A)
   Ratos : ['R1', 'R2', 'R3', 'R4', 'R5', 'R7', 'R8']

Distribuição por condição e trial:
condicao  trial
0.25      T1       7
          T2       8
          T3       6
          T4       6
0.5       T1       6
          T2       5
          T3       5
          T4       5
0.75      T1       7
          T2       8
          T3       8
          T4       6
NOR       T1       6
          T2       7
          T3       7
          T4       6


---
## Fase 3 — Identificação de canais ruins

**O que faz:**  
Aplica dois critérios combinados:
1. **Automático** — RMS < 1 µV (canal morto), kurtosis > 50 (saturação/artefato extremo), std < 0.5 µV
2. **Anotações CSV** — se `acontencimento` contém palavras como 'sinal ruim', 'cabo desconect', 'ruído', marca todos os canais com flag de alerta comportamental

Os canais ruins são **marcados com justificativa** mas não removidos — a remoção ocorre durante a extração de features.

In [7]:
# Palavras-chave nas anotações que indicam sinal comprometido
PALAVRAS_SINAL_RUIM = [
    'sinal ruim', 'signal ruim', 'ruido', 'ruído', 'noise',
    'cabo desconect', 'desconectou', 'desconect',
    'intan precisou', 'intan desconect',
    'sem registro', 'mudanças no sinal', 'signal com ruido',
    'sinal igual',  # saturação
]

def anotacao_indica_sinal_ruim(texto):
    """Retorna True se a anotação comportamental indica problema de sinal."""
    t = str(texto).lower()
    return any(p in t for p in PALAVRAS_SINAL_RUIM)


def identificar_canais_ruins(data_neural, rato, stem, acontencimento=''):
    """Identifica canais ruins combinando critérios automáticos e anotações CSV.

    Parâmetros
    ----------
    data_neural : np.ndarray, shape [N_CANAIS, N_amostras]
    rato        : str
    stem        : str — nome do arquivo sem extensão
    acontencimento : str — anotação do CSV

    Retorna
    -------
    dict: {canal_idx: {'ruim': bool, 'motivo': str, 'fonte': str}}
    """
    resultado = {}
    alerta_csv = anotacao_indica_sinal_ruim(acontencimento)

    for ch in range(data_neural.shape[0]):
        sig = data_neural[ch].astype(float)
        rms = float(np.sqrt(np.mean(sig**2)))
        std = float(np.std(sig))
        kurt = float(kurtosis(sig))

        motivos = []
        fontes  = []

        # Critérios automáticos
        if rms < 1.0:
            motivos.append(f'Canal morto (RMS={rms:.3f} µV < 1 µV)')
            fontes.append('auto')
        if std < 0.5:
            motivos.append(f'Variância nula (std={std:.3f} µV < 0.5 µV)')
            fontes.append('auto')
        if kurt > 50:
            motivos.append(f'Saturação/artefato extremo (kurtosis={kurt:.1f} > 50)')
            fontes.append('auto')

        # Flag do CSV
        if alerta_csv:
            motivos.append(f'Anotação CSV: "{acontencimento}"')
            fontes.append('csv')

        ruim = len(motivos) > 0
        resultado[ch] = {
            'ruim':   ruim,
            'rms':    rms,
            'std':    std,
            'kurt':   kurt,
            'motivo': ' | '.join(motivos) if motivos else '',
            'fonte':  '+'.join(set(fontes)) if fontes else '',
        }

    return resultado


print('✅ Função identificar_canais_ruins() definida.')
print('   Critérios automáticos: RMS < 1 µV | std < 0.5 µV | kurtosis > 50')
print('   Critério CSV: palavras-chave de problema de sinal na coluna "acontencimento"')

✅ Função identificar_canais_ruins() definida.
   Critérios automáticos: RMS < 1 µV | std < 0.5 µV | kurtosis > 50
   Critério CSV: palavras-chave de problema de sinal na coluna "acontencimento"


---
## Fase 4 — Inspeção visual do sinal bruto completo

**O que faz:** Para cada arquivo, gera um painel de 16 subplots (um por canal) com o sinal bruto inteiro.  
Canais ruins são destacados em vermelho com o motivo anotado.  
**Saída:** `EDA_v6_outputs/<stem>/sinal_bruto.png`

In [8]:
def plotar_sinal_bruto(data_neural, t, stem, canais_info, pasta_saida):
    """Plota o sinal bruto completo de todos os canais, destacando os ruins."""
    fig, axes = plt.subplots(N_CANAIS, 1, figsize=(18, N_CANAIS * 1.2), sharex=True)
    fig.suptitle(f'Sinal Bruto Completo — {stem}', fontsize=11, y=1.001)

    for ch, ax in enumerate(axes):
        info  = canais_info.get(ch, {})
        ruim  = info.get('ruim', False)
        cor   = '#e74c3c' if ruim else '#2c3e50'
        alpha = 0.6 if ruim else 0.8
        sig   = data_neural[ch].astype(float)

        ax.plot(t, sig, color=cor, lw=0.3, alpha=alpha)
        label = f'ch{ch+1:02d}'
        if ruim:
            label += f'  ⚠ {info.get("motivo", "ruim")}'
        ax.set_ylabel(label, fontsize=5, rotation=0, ha='right', va='center')
        ax.yaxis.set_tick_params(labelsize=5)
        ax.xaxis.set_tick_params(labelsize=5)
        if ruim:
            ax.set_facecolor('#fff5f5')

    axes[-1].set_xlabel('Tempo (s)', fontsize=8)
    plt.tight_layout()
    caminho = pasta_saida / 'sinal_bruto.png'
    fig.savefig(caminho, dpi=120, bbox_inches='tight')
    plt.close(fig)
    return caminho


print('✅ Função plotar_sinal_bruto() definida.')

✅ Função plotar_sinal_bruto() definida.


---
## Fase 5 — Remoção de ruído 55–65 Hz (filtro Notch)

**O que faz:** Aplica filtro IIR notch centrado em 60 Hz com banda de rejeição 55–65 Hz usando `iirnotch` + `filtfilt` (fase zero).  
**Por que:** Elimina interferência da rede elétrica (60 Hz no Brasil) e suas bandas laterais, sem afetar gamma lento abaixo de 55 Hz nem gamma rápido acima de 65 Hz.

In [9]:
def aplicar_notch(data_neural, fs):
    """Aplica filtro notch 55-65 Hz (centrado em 60 Hz) em todos os canais.

    Usa iirnotch (IIR biquad) + filtfilt (fase zero, sem distorção temporal).
    Operação feita sobre o sinal original (pré-downsample), na FS_ORIGINAL.
    """
    b, a = iirnotch(w0=NOTCH_CENTER, Q=NOTCH_Q, fs=fs)
    data_filtrado = np.zeros_like(data_neural, dtype=float)
    for ch in range(data_neural.shape[0]):
        data_filtrado[ch] = filtfilt(b, a, data_neural[ch].astype(float))
    return data_filtrado


print('✅ Função aplicar_notch() definida.')
print(f'   Notch: {NOTCH_LO}–{NOTCH_HI} Hz | centro={NOTCH_CENTER} Hz | Q={NOTCH_Q:.1f}')

✅ Função aplicar_notch() definida.
   Notch: 55.0–65.0 Hz | centro=60.0 Hz | Q=6.0


---
## Fase 6 — Remoção de DC offset

**O que faz:** Subtrai a média de cada canal (DC offset = componente contínua).  
**Por que:** O sistema Intan pode introduzir offsets de até dezenas de µV. Removê-los antes do downsample evita que o anti-aliasing propague o offset como artefato de baixa frequência.

In [10]:
def remover_dc_offset(data_neural):
    """Remove DC offset subtraindo a média de cada canal."""
    return data_neural - data_neural.mean(axis=1, keepdims=True)


print('✅ Função remover_dc_offset() definida.')

✅ Função remover_dc_offset() definida.


---
## Fase 7 — Downsample + salvamento `sinal_downsample`

**O que faz:**  
1. Anti-aliasing com filtro Butterworth passa-baixa (corte em FS/2 = 500 Hz)  
2. Decimação por fator `DS_FACTOR` (20x: 20 kHz → 1 kHz)  
3. Salva como arquivo `.npy` e metadados `.json`  

**Por que:** LFP relevante está abaixo de 200 Hz. Trabalhar a 1 kHz reduz memória 20x sem perda de informação.

In [11]:
def downsample(data_neural, fs_original, fs_alvo):
    """Anti-aliasing + decimação.  Retorna array [N_CANAIS, N_amostras_down]."""
    up   = fs_alvo
    down = fs_original
    # resample_poly cuida internamente do filtro anti-aliasing
    data_down = np.zeros((data_neural.shape[0],
                          int(data_neural.shape[1] * fs_alvo / fs_original)),
                         dtype=np.float32)
    for ch in range(data_neural.shape[0]):
        data_down[ch] = resample_poly(data_neural[ch].astype(float), up, down).astype(np.float32)
    return data_down


def salvar_downsample(data_down, stem, pasta_saida, metadados=None):
    """Salva o sinal downsampled como .npy + metadados .json."""
    pasta_saida.mkdir(parents=True, exist_ok=True)
    caminho_npy  = pasta_saida / f'{stem}_sinal_downsample.npy'
    caminho_json = pasta_saida / f'{stem}_sinal_downsample.json'

    np.save(caminho_npy, data_down)

    meta = {
        'stem': stem,
        'shape': list(data_down.shape),
        'fs': FS,
        'dtype': str(data_down.dtype),
        'descricao': 'Sinal LFP downsampled [N_CANAIS x N_amostras] float32',
    }
    if metadados:
        meta.update(metadados)

    with open(caminho_json, 'w', encoding='utf-8') as f:
        json.dump(meta, f, ensure_ascii=False, indent=2)

    return caminho_npy


print('✅ Funções downsample() e salvar_downsample() definidas.')

✅ Funções downsample() e salvar_downsample() definidas.


---
## Fase 8 — Conversão para arquivos de sinal por segundo

**O que faz:** Fatia o array downsampled em janelas de 1 segundo (FS amostras cada) e salva como DataFrame Parquet com colunas `t_s, ch00, ch01, …, ch15`.  
**Por que:** Facilita análise por época, cálculo de features por segundo e join com eventos comportamentais pontuais.

In [12]:
def sinal_por_segundo(data_down, stem, pasta_saida, canais_bons=None):
    """Fatia o sinal em janelas de 1 s e salva como Parquet.

    Parâmetros
    ----------
    data_down   : array [N_CANAIS, N_amostras] @ FS
    canais_bons : lista de índices de canais bons (None = todos)
    """
    n_amostras = data_down.shape[1]
    n_segundos = n_amostras // FS

    registros = []
    for s in range(n_segundos):
        inicio = s * FS
        fim    = inicio + FS
        row = {'segundo': s, 't_inicio_s': s, 't_fim_s': s + 1}
        for ch in range(data_down.shape[0]):
            col = f'ch{ch:02d}'
            if canais_bons is None or ch in canais_bons:
                row[col] = data_down[ch, inicio:fim].tolist()
            else:
                row[col] = None
        registros.append(row)

    df_s = pd.DataFrame(registros)
    caminho = pasta_saida / f'{stem}_por_segundo.parquet'
    df_s.to_parquet(caminho, index=False)
    return caminho, n_segundos


print('✅ Função sinal_por_segundo() definida.')

✅ Função sinal_por_segundo() definida.


---
## Fase 9 — Features de estatísticas por canal

**O que faz:** Para cada canal bom e cada segundo do sinal, calcula:
- `mean`, `std`, `rms`, `kurtosis`, `skewness` (temporais)
- `pico_a_pico` (amplitude)
- `pct_outlier_3sigma` (% de amostras > 3σ)

In [13]:
def features_estatisticas(sig):
    """Features temporais básicas de um sinal 1D."""
    mu   = float(sig.mean())
    sd   = float(sig.std())
    rms  = float(np.sqrt(np.mean(sig**2)))
    kurt = float(kurtosis(sig))
    skw  = float(skew(sig))
    p2p  = float(sig.max() - sig.min())
    outlier_pct = float(np.mean(np.abs(sig - mu) > 3 * sd)) * 100
    return {
        'mean': mu, 'std': sd, 'rms': rms,
        'kurtosis': kurt, 'skewness': skw,
        'pico_a_pico': p2p, 'pct_outlier_3s': outlier_pct,
    }


print('✅ Função features_estatisticas() definida.')

✅ Função features_estatisticas() definida.


---
## Fase 10D — Sinal médio por área anatômica + Janelamento do Welch

**Mudança de unidade de análise:** a partir daqui, as análises deixam de ser feitas
por **canal** (16 por rato) e passam a ser feitas por **área anatômica** (a área de
cada canal vem da planilha `planilha_ratos_R1_R7.xlsx`, lida dinamicamente na Fase 0 —
qualquer rato adicionado à planilha, como o R8, entra automaticamente).

- Se uma área tiver 2 canais bons, o sinal usado é a **média ponto-a-ponto** (domínio
  do tempo) dos 2 canais.
- Se a área tiver só 1 canal bom, usa-se esse canal sozinho.
- Canais ruins não entram na média.

Além disso, esta fase refaz manualmente a segmentação do Welch (normalmente interna
ao `scipy.signal.welch`) para poder **salvar cada janela individual**:

- `welch_janelamento/<stem>/<area>_janelas_sinal.npy` — os segmentos brutos no tempo
  (1 s, 50% overlap) usados em cada janela do Welch.
- `welch_janelamento/<stem>/<area>_janelas_psd.npy` — o PSD de cada janela individual
  (antes da média final que dá o PSD Welch "oficial").
- `welch_janelamento/<stem>/<area>_janelas_meta.json` — frequências, tempo de início
  de cada janela e parâmetros usados.


In [14]:
# ── Fase 10D — Sinal médio por área + Janelamento do Welch ────────────────────

WELCH_JANELAMENTO_DIR = Path('welch_janelamento')
WELCH_JANELAMENTO_DIR.mkdir(exist_ok=True)


def slug_area(area):
    """Normaliza um nome de área para uso em nome de arquivo."""
    return re.sub(r'[^0-9a-zA-Z]+', '_', area).strip('_')


def canais_por_area(rato, canais_bons):
    """Agrupa os canais bons por área anatômica.

    Retorna {area: [índices de canais bons naquela área]}.
    Canais sem área definida na planilha (string vazia) são ignorados.
    """
    areas_map = {}
    for ch in canais_bons:
        area = get_area_canal(rato, ch)
        if not area:
            continue
        areas_map.setdefault(area, []).append(ch)
    return areas_map


def sinal_medio_por_area(data_down, rato, canais_bons):
    """Calcula o sinal médio (domínio do tempo) por área anatômica.

    Área com 1 canal bom → retorna esse canal sem alteração.
    Área com >1 canal bom → média ponto-a-ponto dos canais.

    Retorna
    -------
    sinais_area   : {area: array 1D}
    areas_canais  : {area: [índices de canais usados]}  (para registrar no relatório)
    """
    areas_canais = canais_por_area(rato, canais_bons)
    sinais_area = {}
    for area, chs in areas_canais.items():
        if len(chs) == 1:
            sinais_area[area] = data_down[chs[0]].astype(float)
        else:
            sinais_area[area] = data_down[chs].astype(float).mean(axis=0)
    return sinais_area, areas_canais


def welch_com_janelas(sig, fs=FS, nperseg=WELCH_NPERSEG,
                       noverlap=WELCH_NOVERLAP, nfft=WELCH_NFFT,
                       window=WELCH_WINDOW):
    """Refaz manualmente a segmentação do Welch para expor cada janela individual.

    Replica exatamente o que scipy.signal.welch(..., scaling='density',
    detrend='constant') faz internamente, mas guarda cada janela em vez de
    descartá-la após a média.

    Retorna
    -------
    freqs         : array [n_freqs] — frequências (Hz)
    psd_medio     : array [n_freqs] — PSD final (média das janelas), igual ao
                    retornado por scipy.signal.welch
    janelas_sinal : array [n_janelas, nperseg] — segmentos BRUTOS no tempo
                    (sem detrend/janela de Hanning aplicada — sinal como está)
    janelas_psd   : array [n_janelas, n_freqs] — PSD de cada janela individual
    t_inicio      : array [n_janelas] — tempo de início (s) de cada janela
    """
    from scipy.signal import get_window
    from scipy.fft import rfft, rfftfreq

    sig = np.asarray(sig, dtype=float)
    n = len(sig)
    nperseg = min(nperseg, n)
    passo = nperseg - noverlap
    if passo <= 0:
        passo = nperseg

    inicios = list(range(0, n - nperseg + 1, passo))
    if not inicios:
        inicios = [0]
        nperseg = n

    win = get_window(window, nperseg)
    scale = 1.0 / (fs * (win ** 2).sum())
    freqs = rfftfreq(nfft, d=1.0 / fs)

    janelas_sinal = np.zeros((len(inicios), nperseg))
    janelas_psd = np.zeros((len(inicios), len(freqs)))

    for i, ini in enumerate(inicios):
        seg = sig[ini:ini + nperseg]
        janelas_sinal[i] = seg

        seg_d = seg - seg.mean()  # detrend='constant' (default do scipy.welch)
        seg_w = seg_d * win
        Xf = rfft(seg_w, n=nfft)
        psd_i = (np.abs(Xf) ** 2) * scale
        if nfft % 2 == 0:
            psd_i[1:-1] *= 2
        else:
            psd_i[1:] *= 2
        janelas_psd[i] = psd_i

    psd_medio = janelas_psd.mean(axis=0)
    t_inicio = np.array(inicios) / fs

    return freqs, psd_medio, janelas_sinal, janelas_psd, t_inicio


def salvar_janelamento_welch(sinais_area, stem, pasta_saida_base=WELCH_JANELAMENTO_DIR):
    """Calcula e salva o janelamento do Welch para cada área de um arquivo.

    Para cada área, salva em <pasta_saida_base>/<stem>/:
      - <area>_janelas_sinal.npy  → [n_janelas, nperseg]  segmentos brutos no tempo
      - <area>_janelas_psd.npy    → [n_janelas, n_freqs]  PSD de cada janela
      - <area>_janelas_meta.json  → freqs, t_inicio, parâmetros do Welch

    Retorna {area: n_janelas} como resumo.
    """
    pasta = pasta_saida_base / stem
    pasta.mkdir(parents=True, exist_ok=True)

    resumo = {}
    for area, sig in sinais_area.items():
        area_id = slug_area(area)

        freqs, psd_medio, janelas_sinal, janelas_psd, t_inicio = welch_com_janelas(sig)

        np.save(pasta / f'{area_id}_janelas_sinal.npy', janelas_sinal)
        np.save(pasta / f'{area_id}_janelas_psd.npy', janelas_psd)

        meta = {
            'area': area,
            'n_janelas': int(janelas_sinal.shape[0]),
            'nperseg': int(janelas_sinal.shape[1]),
            'fs': FS,
            'nfft': WELCH_NFFT,
            'noverlap': WELCH_NOVERLAP,
            'window': WELCH_WINDOW,
            'freqs_hz': freqs.tolist(),
            't_inicio_s': t_inicio.tolist(),
        }
        (pasta / f'{area_id}_janelas_meta.json').write_text(
            json.dumps(meta, ensure_ascii=False, indent=2)
        )
        resumo[area] = janelas_sinal.shape[0]

    return resumo


print('✅ Fase 10D definida: sinal médio por área + janelamento do Welch.')
print(f'   Saída: {WELCH_JANELAMENTO_DIR}/<stem>/<area>_janelas_sinal.npy + _janelas_psd.npy + _janelas_meta.json')


✅ Fase 10D definida: sinal médio por área + janelamento do Welch.
   Saída: welch_janelamento/<stem>/<area>_janelas_sinal.npy + _janelas_psd.npy + _janelas_meta.json


---
## Fase 10A — PSD Welch + bandas espectrais

**O que faz:**  
Estimativa de PSD usando janela de Hanning, **1 s** de segmento, 50% de overlap e NFFT=1024.  
Bandas de interesse fisiológico:
- **Delta** (1–4 Hz)
- **Theta** (6–12 Hz)
- **Beta** (13–30 Hz)
- **Gamma Lento** (25–55 Hz)
- **Gamma Rápido** (65–110 Hz)

**Por que Welch com Hanning e janela de 1 s?**  
A janela de Hanning minimiza vazamento espectral em sinais LFP com picos estreitos como theta.

### Justificativa da janela de 1 segundo

Se você seguir a lógica do texto:

* A menor frequência de interesse é **7 Hz**.
* O período de uma onda de 7 Hz é:

$$T = \frac{1}{f} = \frac{1}{7} \approx 0{,}143 \text{ s}$$

ou seja, aproximadamente **143 ms**.

Portanto:

* **Mínimo teórico:** o segmento deveria ter pelo menos **143 ms** para conter um ciclo completo de 7 Hz.
* **Recomendação prática:** usar vários ciclos. Por exemplo:

  * 3 ciclos → $3/7 \approx 0{,}43$ s
  * 5 ciclos → $5/7 \approx 0{,}71$ s
  * 7 ciclos → $7/7 = 1$ s

Assim, para analisar frequências a partir de **7 Hz**, um segmento de **1 segundo** já contém **7 ciclos completos** dessa frequência e costuma ser uma escolha razoável para FFT, desde que a estacionariedade do sinal seja mantida.

Além disso, a resolução espectral da FFT é dada por:

$$\Delta f = \frac{1}{T_{\text{janela}}}$$

Então:

* janela de 1 s → resolução de **1 Hz**;
* janela de 2 s → resolução de 0,5 Hz;
* janela de 4 s → resolução de 0,25 Hz.

Logo, se você precisa distinguir bem componentes próximas de 7 Hz (por exemplo, 7 Hz vs. 7,5 Hz), pode ser interessante usar janelas maiores que 1 s, desde que o sinal continue aproximadamente estacionário.

> ⚠️ **Harmônicas de Theta não são bandas fixas neste pipeline.**  
> Elas são calculadas dinamicamente na **Fase 10C** a partir do pico theta real de cada registro,  
> e usadas exclusivamente como **controle metodológico** do PAC (Fase 12).

> ℹ️ A potência por banda calculada aqui (Fase 10A) resume o espectro em 5 faixas fixas.  
> A **Fase 10B**, logo a seguir, complementa isso com uma frequência única e contínua
> por segundo/canal (centroide espectral) — útil para comparar ratos com sessões de
> durações diferentes sem depender de bandas discretas.


In [15]:
def psd_welch_bandas(sig, fs=FS):
    """PSD Welch (Hanning, 1s, 50%, NFFT=1024) + potência por banda."""
    nperseg  = min(WELCH_NPERSEG, len(sig))
    noverlap = nperseg // 2

    freqs, psd = welch(
        sig, fs=fs,
        window=WELCH_WINDOW,
        nperseg=nperseg,
        noverlap=noverlap,
        nfft=WELCH_NFFT,
        scaling='density',
    )

    potencias = {}
    for nome, (f1, f2) in BANDAS.items():
        mask = (freqs >= f1) & (freqs < f2)
        potencias[f'pot_{nome}'] = float(psd[mask].mean()) if mask.any() else np.nan

    theta = potencias.get('pot_theta', np.nan)
    gamma_lento = potencias.get('pot_gamma_lento', np.nan)
    gamma_rapido = potencias.get('pot_gamma_rapido', np.nan)

    potencias['theta_gamma_lento_ratio'] = (
        theta / gamma_lento
        if np.isfinite(theta) and np.isfinite(gamma_lento) and gamma_lento > 0
        else np.nan
    )

    potencias['theta_gamma_rapido_ratio'] = (
        theta / gamma_rapido
        if np.isfinite(theta) and np.isfinite(gamma_rapido) and gamma_rapido > 0
        else np.nan
    )

    delta = potencias.get('pot_delta', np.nan)
    potencias['delta_theta_ratio'] = (
        delta / theta
        if np.isfinite(delta) and np.isfinite(theta) and theta > 0
        else np.nan
    )
    # Entropia espectral normalizada
    p_norm = psd / (psd.sum() + 1e-12)
    potencias['entropia_espectral'] = float(
        -np.sum(p_norm * np.log2(p_norm + 1e-12)) / np.log2(len(p_norm))
    )

    return freqs, psd, potencias


def plotar_psd(freqs, psds_por_area, areas, stem, pasta_saida, rato=None):
    """Plot PSD Welch de todas as áreas em escala log (visão geral)."""
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    cmap = plt.get_cmap('tab20', max(len(areas), 1))

    for i, area in enumerate(areas):
        freqs_a, psd_a = psds_por_area[area]
        for ax in axes:
            ax.semilogy(freqs_a, psd_a, lw=0.8, alpha=0.7,
                        color=cmap(i), label=area)

    for ax, xlim in zip(axes, [(0, 120), (0, 40)]):
        ax.set_xlim(*xlim)
        ax.axvspan(NOTCH_LO, NOTCH_HI, color='red', alpha=0.08, label='Notch 55-65 Hz')
        for nome, (f1, f2) in BANDAS.items():
            ax.axvspan(f1, f2, alpha=0.06, color=CORES_BANDAS[nome], label=nome)
        ax.set_xlabel('Frequência (Hz)')
        ax.set_ylabel('PSD (µV²/Hz)')
        ax.grid(True, alpha=0.2)

    axes[0].set_title(f'PSD Welch — {stem}  (0–120 Hz)', fontsize=9)
    axes[1].set_title('Zoom 0–40 Hz', fontsize=9)

    handles, labels = axes[0].get_legend_handles_labels()
    by_label = dict(zip(labels, handles))
    axes[0].legend(by_label.values(), by_label.keys(),
                   fontsize=6, ncol=2, loc='upper right')

    plt.suptitle(f'Welch Hanning 1s 50% NFFT={WELCH_NFFT} — {stem} (por área)', fontsize=10)
    plt.tight_layout()
    caminho = pasta_saida / 'psd_welch.png'
    fig.savefig(caminho, dpi=150, bbox_inches='tight')
    plt.close(fig)
    return caminho


def plotar_psd_por_canal(freqs, psds_por_area, areas, stem, pasta_saida, rato=None):
    """Plot PSD Welch individual por área anatômica.

    Gera um subplot por área (sinal já é a média dos canais bons daquela área),
    organizado em grade de 4 colunas.
    Salva em: <pasta_saida>/psd_welch_por_canal.png
    """
    n_areas = len(areas)
    if n_areas == 0:
        return None

    n_cols = 4
    n_rows = int(np.ceil(n_areas / n_cols))
    fig, axes = plt.subplots(n_rows, n_cols,
                             figsize=(5 * n_cols, 3.5 * n_rows),
                             squeeze=False)

    cmap = plt.get_cmap('tab20', max(n_areas, 1))

    for idx, area in enumerate(areas):
        row, col = divmod(idx, n_cols)
        ax = axes[row][col]

        freqs_a, psd_a = psds_por_area[area]

        ax.semilogy(freqs_a, psd_a, lw=1.0, color=cmap(idx))

        # Sombrear bandas
        for nome, (f1, f2) in BANDAS.items():
            ax.axvspan(f1, f2, alpha=0.10, color=CORES_BANDAS[nome], label=nome)
        ax.axvspan(NOTCH_LO, NOTCH_HI, color='red', alpha=0.08)

        ax.set_xlim(0, 120)
        ax.set_xlabel('Hz', fontsize=7)
        ax.set_ylabel('PSD', fontsize=7)
        ax.tick_params(labelsize=6)
        ax.grid(True, alpha=0.2)
        ax.set_title(area, fontsize=8, fontweight='bold')

    # Esconder axes extras
    for idx in range(n_areas, n_rows * n_cols):
        row, col = divmod(idx, n_cols)
        axes[row][col].set_visible(False)

    # Legenda de bandas em um dos axes visíveis
    handles, labels = axes[0][0].get_legend_handles_labels()
    by_label = dict(zip(labels, handles))
    fig.legend(by_label.values(), by_label.keys(),
               loc='lower right', fontsize=7, ncol=len(BANDAS))

    plt.suptitle(f'PSD Welch por Área — {stem}  (Hanning 1s, 50%, NFFT={WELCH_NFFT})',
                 fontsize=11, y=1.01)
    plt.tight_layout()
    caminho = pasta_saida / 'psd_welch_por_canal.png'
    fig.savefig(caminho, dpi=130, bbox_inches='tight')
    plt.close(fig)
    return caminho


print('✅ Funções psd_welch_bandas(), plotar_psd() e plotar_psd_por_canal() definidas.')
print('   Bandas: delta(1-4Hz), theta(6-12Hz), beta(13-30Hz), gamma_lento(25-55Hz), gamma_rapido(65-110Hz)')
print('   Janela Welch: 1 s | NFFT=1024 | plots agora recebem psds_por_area / lista de áreas')

✅ Funções psd_welch_bandas(), plotar_psd() e plotar_psd_por_canal() definidas.
   Bandas: delta(1-4Hz), theta(6-12Hz), beta(13-30Hz), gamma_lento(25-55Hz), gamma_rapido(65-110Hz)
   Janela Welch: 1 s | NFFT=1024 | plots agora recebem psds_por_area / lista de áreas


---
## Fase 10B — Centroide Espectral (frequência contínua por canal)

**O que faz:**
A partir do mesmo PSD Welch calculado na Fase 10A (Hanning, 1 s, 50% overlap, NFFT=1024),
calcula o **centroide espectral** — a frequência média do sinal, ponderada pela potência
em cada frequência:

$$f_{centroide} = \frac{\sum_i f_i \cdot P(f_i)}{\sum_i P(f_i)}$$

Diferente das potências por banda (Fase 10A), que resumem o espectro em 5 valores
categóricos fixos, o centroide devolve **um único número contínuo em Hz** por
segundo/canal — a "frequência dominante" daquele instante do sinal.

**Por que:**
Bandas fixas (delta, theta, beta...) são úteis para hipóteses fisiológicas específicas,
mas descartam informação de onde exatamente dentro da banda (ou entre bandas) a energia
está concentrada. O centroide espectral preserva essa informação como uma série temporal
contínua, o que permite:

- Comparar a "frequência típica" do sinal entre ratos e sessões sem depender de limiares
  de banda arbitrários;
- Construir, junto com a normalização temporal (0–100% da sessão), uma curva de frequência
  comparável entre ratos com gravações de durações diferentes — já que o centroide por
  segundo pode ser reamostrado para um eixo comum de tempo relativo.

O centroide é calculado limitado à faixa 1–110 Hz (mesma faixa coberta pelas bandas de
interesse), para não ser distorcido por ruído de baixa frequência nem pela região do
notch (55–65 Hz) ou acima de 110 Hz.


In [16]:
def centroide_espectral(freqs, psd, f_min=1, f_max=110):
    """Centroide espectral (frequência média ponderada pela potência).

    Resume o PSD Welch de um segmento em um único valor contínuo em Hz,
    complementando as potências por banda calculadas na Fase 10A.

    Parâmetros
    ----------
    freqs : array — frequências do PSD (Hz), retornado por psd_welch_bandas()
    psd   : array — densidade espectral de potência correspondente
    f_min, f_max : float — faixa de frequência considerada no cálculo
                   (default 1-110 Hz, mesma faixa das bandas de interesse)

    Retorna
    -------
    float — frequência do centroide espectral (Hz), ou NaN se não houver
            potência na faixa considerada
    """
    mask = (freqs >= f_min) & (freqs <= f_max)
    f = freqs[mask]
    p = psd[mask]
    if p.sum() <= 0:
        return float('nan')
    return float((f * p).sum() / p.sum())


print('✅ Função centroide_espectral() definida.')
print(f'   Faixa considerada: 1–110 Hz | usa o mesmo PSD Welch da Fase 10A')
print('   Saída: 1 valor contínuo em Hz por segundo/canal (coluna ch<N>_centroide_hz)')


✅ Função centroide_espectral() definida.
   Faixa considerada: 1–110 Hz | usa o mesmo PSD Welch da Fase 10A
   Saída: 1 valor contínuo em Hz por segundo/canal (coluna ch<N>_centroide_hz)


---
## Fase 10C — Detecção do pico Theta real + harmônicas como controle

**Motivação metodológica:**  
Uma oscilação Theta não senoidal naturalmente produz harmônicas matemáticas (2×, 3×, 4× do pico fundamental). Se o pico Theta estiver em 8 Hz, haverá energia em ~16, ~24, ~32, ~40 Hz mesmo sem nenhuma oscilação fisiológica nessas frequências. Isso pode criar um PAC *aparente* entre Theta (fase) e Slow Gamma (amplitude) quando na verdade o sinal em 25–32 Hz é apenas a 3ª harmônica do Theta.

**O que esta fase faz:**
1. Localiza o pico espectral dominante dentro da banda theta (6–12 Hz) via PSD Welch
2. Calcula h2, h3, h4 a partir desse pico real (não de faixas fixas)
3. Verifica se alguma harmônica cai dentro de Slow Gamma (25–55 Hz)
4. Gera um **flag de alerta** por registro para uso na interpretação do PAC
5. Salva um plot do espectro anotado com as posições das harmônicas



In [17]:
def detectar_pico_theta(sig, fs=FS,
                        banda_theta=(6, 12)):
    """Encontra o pico espectral dominante dentro da banda theta.

    Retorna
    -------
    theta_peak_hz : float — frequência do pico (Hz)
    theta_peak_psd : float — potência no pico (µV²/Hz)
    freqs, psd    : arrays completos para uso externo
    """
    nperseg = min(WELCH_NPERSEG, len(sig))
    freqs, psd = welch(
        sig, fs=fs,
        window=WELCH_WINDOW,
        nperseg=nperseg,
        noverlap=nperseg // 2,
        nfft=WELCH_NFFT,
        scaling='density',
    )
    mask_theta = (freqs >= banda_theta[0]) & (freqs <= banda_theta[1])
    psd_theta  = psd[mask_theta]
    freqs_theta = freqs[mask_theta]

    idx_pico = np.argmax(psd_theta)
    return (
        float(freqs_theta[idx_pico]),  # frequência do pico
        float(psd_theta[idx_pico]),    # potência do pico
        float(psd_theta.mean()),       # potência média theta
        freqs,
        psd
    )

def calcular_harmonicas(theta_peak_hz, n_harmonicas=4):
    """Calcula as n primeiras harmônicas do pico theta.

    Retorna lista de (ordem, frequência_hz).
    A fundamental (h1) é o próprio pico theta.
    """
    return [(n, round(n * theta_peak_hz, 2))
            for n in range(2, n_harmonicas + 2)]  # h2, h3, h4, h5

def existe_pico_espectral(
        freqs,
        psd,
        freq_alvo,
        tolerancia_hz=1.0,
        fator_minimo=1.5):
    """
    Verifica se existe um pico espectral real próximo
    de uma frequência alvo.

    Parâmetros
    ----------
    freqs : array
    psd : array
    freq_alvo : float
        Frequência da harmônica esperada.
    tolerancia_hz : float
        Janela de busca ao redor da frequência.
    fator_minimo : float
        Pico deve ser pelo menos X vezes maior
        que a média local.

    Retorna
    -------
    existe : bool
    pico_freq : float
    pico_psd : float
    """

    mask = (
        (freqs >= freq_alvo - tolerancia_hz) &
        (freqs <= freq_alvo + tolerancia_hz)
    )

    if not np.any(mask):
        return False, np.nan, np.nan

    freqs_local = freqs[mask]
    psd_local = psd[mask]

    idx = np.argmax(psd_local)

    pico_freq = float(freqs_local[idx])
    pico_psd = float(psd_local[idx])

    media_local = float(np.mean(psd_local))

    existe = pico_psd > fator_minimo * media_local

    return existe, pico_freq, pico_psd

def plotar_espectro_harmonicas(freqs, psd, theta_peak_hz, harmonicas,
                               alerta, stem, area, pasta_saida):
    """Plota o espectro 0-120 Hz com marcação do pico theta e suas harmônicas.

    `area` é o rótulo (string) da área anatômica (ex: 'CA3b'), já que a
    análise passou a ser feita por área em vez de por canal.
    """
    fig, ax = plt.subplots(figsize=(13, 4))
    ax.semilogy(freqs, psd, lw=0.9, color='#2c3e50', alpha=0.85)

    # Pico theta fundamental
    ax.axvline(theta_peak_hz, color='#3498db', lw=1.5, ls='-',
               label=f'θ pico = {theta_peak_hz:.1f} Hz')

    # Harmônicas
    cores_harm = ['#e74c3c', '#e67e22', '#9b59b6', '#1abc9c']
    for (ordem, freq_h), cor in zip(harmonicas, cores_harm):
        ax.axvline(freq_h, color=cor, lw=1.2, ls='--', alpha=0.85,
                   label=f'h{ordem} = {freq_h:.1f} Hz')

    # Sombreamento das bandas de interesse
    ax.axvspan(*BANDAS['gamma_lento'],  alpha=0.07,
               color=CORES_BANDAS['gamma_lento'],  label='Gamma Lento')
    ax.axvspan(*BANDAS['gamma_rapido'], alpha=0.07,
               color=CORES_BANDAS['gamma_rapido'], label='Gamma Rápido')

    titulo = (f'Espectro + Harmônicas de θ — {stem} | {area}\n'
              f'θ pico = {theta_peak_hz:.1f} Hz  |  '
              + ('⚠️ ALERTA: harmônica em Gamma Lento' if alerta
                 else '✅ Sem harmônicas em Gamma Lento'))
    ax.set_title(titulo, fontsize=9,
                 color='#c0392b' if alerta else '#27ae60')
    ax.set_xlim(0, 120)
    ax.set_xlabel('Frequência (Hz)', fontsize=8)
    ax.set_ylabel('PSD (µV²/Hz)', fontsize=8)
    ax.legend(fontsize=6, ncol=3, loc='upper right')
    ax.grid(True, alpha=0.2)

    plt.tight_layout()
    caminho = pasta_saida / f'harmonicas_theta_{slug_area(area)}.png'
    fig.savefig(caminho, dpi=130, bbox_inches='tight')
    plt.close(fig)
    return caminho


def verificar_alerta_pac(
        harmonicas,
        freqs,
        psd,
        banda_gamma_lento=(25, 55),
        tolerancia_hz=None):

    tol = (
        tolerancia_hz
        if tolerancia_hz is not None
        else HARMONICA_TOLERANCIA_HZ
    )

    f1, f2 = banda_gamma_lento

    detalhes = []

    for ordem, freq_h in harmonicas:

        existe, pico_freq, pico_psd = existe_pico_espectral(
            freqs,
            psd,
            freq_h,
            tolerancia_hz=tol
        )

        if not existe:
            continue

        if (f1 - tol) <= pico_freq <= (f2 + tol):

            detalhes.append({
                'ordem': ordem,
                'freq_teorica_hz': freq_h,
                'freq_pico_hz': pico_freq,
                'pico_psd': pico_psd,
                'banda': 'gamma_lento',
                'mensagem':
                    f'Harmônica h{ordem} detectada em '
                    f'{pico_freq:.2f} Hz '
                    f'(esperado: {freq_h:.2f} Hz). '
                    f'Possível confundidor para '
                    f'PAC Theta–Slow Gamma.'
            })

    alerta = len(detalhes) > 0

    return alerta, detalhes


print('✅ Fase 10C definida: detecção dinâmica de pico θ + controle de harmônicas.')
print(f'   Tolerância para alerta: ±{HARMONICA_TOLERANCIA_HZ} Hz')
print('   Saída: harmonicas_theta_ch<N>.png + flag alerta por registro')

✅ Fase 10C definida: detecção dinâmica de pico θ + controle de harmônicas.
   Tolerância para alerta: ±3.0 Hz
   Saída: harmonicas_theta_ch<N>.png + flag alerta por registro


In [18]:
def wavelet_morlet(sig, fs, freqs_alvo=None, w=8.0):
    """CWT de Morlet para um sinal 1D."""
    if freqs_alvo is None:
        freqs_alvo = np.logspace(np.log10(4), np.log10(150), 120)

    # O nome da wavelet determina a largura de banda (1.5) e freq central (w)
    wavelet_name = f'cmor1.5-{w}'

    # CORREÇÃO: Fórmula exata para PyWavelets sem o 2*pi
    escalas = (w * fs) / freqs_alvo

    # O cwt já retorna as frequências equivalentes no segundo argumento 
    coef, freq_reais = pywt.cwt(sig, scales=escalas, wavelet=wavelet_name,
                                sampling_period=1.0 / fs, method='fft')
    
    power = np.abs(coef) ** 2
    return freqs_alvo, power

import os
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path

def plotar_escalogramas_por_banda(freqs_alvo, power, t, stem, canal, pasta_saida,
                                  tmax_plot=30):
    pasta_wavelet = Path(pasta_saida) / 'wavelet_escalograma'  # ← removido / stem
    pasta_wavelet.mkdir(parents=True, exist_ok=True)
    
    n_pts  = min(int(tmax_plot * FS), power.shape[1])
    t_plot = t[:n_pts]
    caminhos_salvos = []
    
    for nome_banda, (f1, f2) in BANDAS.items():
        idx_banda = (freqs_alvo >= f1) & (freqs_alvo <= f2)
        if not np.any(idx_banda):
            continue

        freqs_banda = freqs_alvo[idx_banda]
        pw_plot     = power[idx_banda, :n_pts]

        fig, ax = plt.subplots(figsize=(14, 4))
        im = ax.pcolormesh(t_plot, freqs_banda, 10 * np.log10(pw_plot + 1e-12),
                           shading='auto', cmap='inferno')
        plt.colorbar(im, ax=ax, label='Potência (dB)')

        ax.set_ylabel('Frequência (Hz)', fontsize=8)
        ax.set_xlabel('Tempo (s)', fontsize=8)
        ax.set_yscale('log' if (f2 - f1) > 20 else 'linear')
        ax.set_ylim(f1, f2)
        ax.set_title(
            f'Wavelet Morlet ({nome_banda} {f1}-{f2}Hz) — {stem} | ch{canal+1:02d} (primeiros {tmax_plot}s)',
            fontsize=9
        )
        plt.tight_layout()

        caminho_final = pasta_wavelet / f'wavelet_{nome_banda.lower()}_ch{canal+1:02d}.png'
        fig.savefig(caminho_final, dpi=120, bbox_inches='tight')
        plt.close(fig)
        caminhos_salvos.append(caminho_final)

    return caminhos_salvos
print('✅ Funções wavelet_morlet() e plotar_escalograma() definidas.')

✅ Funções wavelet_morlet() e plotar_escalograma() definidas.


---
## Fase 11 — PAC

---

### O que é o PAC e por que ele importa aqui

O Phase-Amplitude Coupling (PAC) mede o acoplamento entre
a **fase** de uma oscilação lenta e a **amplitude** de uma
oscilação rápida. No hipocampo, o PAC Theta–Gamma reflete
como o ciclo theta (~125 ms) organiza temporalmente os
bursts de gamma: diferentes grupos neuronais disparam em
fases distintas do theta, e a amplitude do gamma em cada
fase carrega representações específicas de memória
(Buzsáki, 2002; Colgin, 2016).

No seu projeto isso é relevante porque Neves et al. (2022)
demonstraram, em ratos Wistar executando uma tarefa NOR
com deslocamento de objetos, que o PAC Theta–Fast Gamma
foi significativamente maior durante o teste em que os
ratos discriminaram o objeto deslocado (HD) versus quando
não discriminaram (LD) — especificamente no giro denteado
esquerdo (t(5) = 3.856, p = 0.012, d' = 1.074). Ou seja:
o PAC captura algo que a potência de banda isolada não
captura. Esse é o motivo de calculá-lo no seu pipeline.

---

### Motivação metodológica para o controle de harmônicas

Uma oscilação Theta **não senoidal** produz harmônicas
matemáticas (h2, h3, h4… = 2×, 3×, 4× do pico fundamental)
mesmo sem nenhuma oscilação fisiológica nessas frequências.
Se o pico Theta estiver em 8 Hz, haverá energia em ~16,
~24, ~32, ~40 Hz por matemática pura da forma de onda.

Isso cria um PAC *aparente* entre Theta (fase) e
Slow Gamma (25–55 Hz): o MI calculado sobe não porque há
acoplamento fisiológico real, mas porque a 3ª harmônica
do theta (~24–32 Hz) cai dentro da banda de slow gamma
e é modulada pela própria fase theta — trivialmente,
porque é a mesma oscilação.

O PAC Theta–Fast Gamma (65–110 Hz) é muito menos
suscetível porque mesmo h8 de um theta em 8 Hz estaria
em 64 Hz (borda inferior), e picos harmônicos decaem
rapidamente em amplitude com a ordem. Por isso o PAC
Theta–Fast Gamma é a **hipótese principal** do pipeline,
seguindo Neves et al. (2022) e Fernández-Ruiz et al. (2021).

O controle de harmônicas segue a recomendação de
Scheffer-Teixeira & Tort (2016), procedimento adotado
por Neves et al. (2022).

---

### O que esta fase faz

1. Localiza o **pico espectral dominante** dentro da banda
   theta (6–12 Hz) via PSD Welch — usando o pico *real*
   do sinal, não uma frequência fixa
2. Calcula h2, h3, h4, h5 a partir desse pico real
3. Verifica se alguma harmônica apresenta um pico
   espectral real próximo (tolerância ±3 Hz) dentro de
   Slow Gamma (25–55 Hz)
4. Gera um **flag** `pac_sl_alerta_harmonicas` por canal
   por registro — salvo em `features.parquet`
5. Salva o espectro anotado com as posições das harmônicas
   em `harmonicas_theta_ch<N>.png`

---

### Para que o flag é usado — o que ele decide no projeto

O flag **não exclui registros nem frequências**. Ele é uma
**ferramenta de decisão analítica** com três usos concretos:

**1. Decidir qual PAC reportar como resultado principal**

```python
pct_alerta = df.groupby('condicao')[
    'ch01_pac_sl_alerta_harmonicas'
].mean() * 100
print(pct_alerta)
```

- Se `alerta = True` em mais de 40% dos registros →
  o PAC Theta–Fast Gamma é a feature principal para o ML
  e para a narrativa do TCC. O PAC Theta–Slow Gamma é
  reportado com nota metodológica.
- Se `alerta = False` na maioria → ambos os PACes são
  analisados. Aí o interessante é verificar se slow e
  fast gamma se comportam diferente entre as seções, o
  que a literatura prediz: slow gamma reflete recuperação
  de memória (CA3→CA1, objeto familiar) e fast gamma
  reflete codificação de novidade (MEC→CA1, objeto
  deslocado) — Colgin et al. (2009); Colgin (2016).

**2. Investigar se o flag varia entre ratos ou seções**

Se `alerta = True` for mais frequente em uma seção
específica, isso em si é um achado: significa que o theta
dos ratos fica mais não-senoidal naquele momento da
sessão, o que tem interpretação biológica — theta mais
não-senoidal está associado a maior engajamento cognitivo
(Buzsáki, 2002).


**3. Escrever a metodologia com respaldo publicado**

O flag permite escrever no TCC:

> *"Para controlar possíveis efeitos decorrentes da
> não-senoidalidade da oscilação Theta, os picos
> espectrais correspondentes às harmônicas do pico Theta
> dominante foram identificados individualmente em cada
> registro, seguindo Scheffer-Teixeira & Tort (2016) e
> Neves et al. (2022). Registros nos quais uma harmônica
> de ordem h3 ou h4 coincidia com a faixa de Slow Gamma
> (25–55 Hz, tolerância ±3 Hz) foram sinalizados com flag
> de alerta e interpretados com cautela adicional. O PAC
> Theta–Fast Gamma (65–110 Hz) foi adotado como hipótese
> principal por ser robusto contra esse confundidor."*

---

### Fluxo de decisão

In [19]:
from scipy.signal import hilbert


def bandpass_butter(sig, fs, f_low, f_high, order=4):
    """Filtro Butterworth passa-banda, fase zero (sosfiltfilt)."""
    sos = butter(order, [f_low, f_high], btype='bandpass',
                 fs=fs, output='sos')
    return sosfiltfilt(sos, sig)


def modulation_index(fase, amplitude, n_bins=18):
    """Índice de Modulação (MI) de Tort et al. 2010 corrigido.
    
    Normaliza a divergência de Kullback-Leibler pelo log(N) para quantificar
    o acoplamento fase-amplitude de maneira robusta.
    """
    # 1. Definição dos limites dos bins (-pi a pi)
    bins = np.linspace(-np.pi, np.pi, n_bins + 1)
    amp_por_fase = np.zeros(n_bins)
    
    # 2. Distribuição da amplitude nos bins de fase
    for i in range(n_bins):
        # Usar <= no limite superior do último bin para não perder o valor de pi
        if i == n_bins - 1:
            mask = (fase >= bins[i]) & (fase <= bins[i + 1])
        else:
            mask = (fase >= bins[i]) & (fase < bins[i + 1])
            
        amp_por_fase[i] = amplitude[mask].mean() if mask.any() else 0.0

    # 3. Transformar em distribuição de probabilidade empírica (P)
    sum_amp = amp_por_fase.sum()
    if sum_amp == 0:
        return 0.0, np.ones(n_bins) / n_bins
        
    p = amp_por_fase / sum_amp
    q = np.ones(n_bins) / n_bins  # Distribuição uniforme teórica
    
    # 4. Cálculo estrito da Divergência KL (apenas onde p > 0)
    # Evita distorções causadas por somas artificiais de ruído numérico (1e-12)
    mask_p = p > 0
    dkl = np.sum(p[mask_p] * np.log(p[mask_p] / q[mask_p]))
    
    # 5. Normalização pelo log(N)
    mi = dkl / np.log(n_bins)
    
    return float(mi), p


def calcular_pac(sig, fs=FS,
                 banda_fase=(6, 12),
                 banda_amp=(25, 55),
                 n_bins=18):
    """Calcula PAC (MI) entre fase de banda_fase e amplitude de banda_amp."""
    sig_fase = bandpass_butter(sig, fs, *banda_fase)
    sig_amp  = bandpass_butter(sig, fs, *banda_amp)

    # A fase deve ser extraída do sinal analítico da banda lenta
    fase      = np.angle(hilbert(sig_fase))
    # A amplitude (envelope) é obtida pelo módulo do sinal analítico da banda rápida
    amplitude = np.abs(hilbert(sig_amp))

    mi, amp_por_fase = modulation_index(fase, amplitude, n_bins)
    
    # Ajuste dos centros dos bins para alinhamento correto no plot
    bins_centro = np.linspace(-np.pi + np.pi / n_bins,
                              np.pi  - np.pi / n_bins, n_bins)
    
    return mi, amp_por_fase, bins_centro


def plotar_pac(amp_por_fase, bins_centro, mi, stem, area,
               banda_fase, banda_amp, pasta_saida,
               rato=None, alerta_harmonicas=False):
    """Comodulogram PAC: painel polar + painel cartesiano lado a lado.

    Layout inspirado na Figura 4 do artigo (Neves et al. 2022):
      Painel esquerdo  -- grafico polar: amplitude media por fase (rosa de ventos)
      Painel direito   -- grafico cartesiano: amplitude vs fase (0-360 graus)
                          com linha de referencia uniforme e preenchimento de area

    Parametros
    ----------
    amp_por_fase        : distribuicao normalizada de amplitude por bin de fase
    bins_centro         : centros dos bins (radianos)
    mi                  : indice de modulacao (float)
    stem / canal        : identificadores do registro e eletrodo
    banda_fase/banda_amp: tuplas (f_low, f_high) em Hz
    pasta_saida         : Path de saida
    rato                : str -- usado para lookup anatomico
    alerta_harmonicas   : bool -- True ativa aviso de harmonica
    """
    area_str = area
    n_bins_plt = len(amp_por_fase)
    nivel_uniforme = 1.0 / n_bins_plt

    # Fecha o circulo para o polar
    theta_p = np.append(bins_centro, bins_centro[0])
    r_p     = np.append(amp_por_fase, amp_por_fase[0])

    # Eixo cartesiano: converter radianos -> graus
    graus   = np.degrees(bins_centro)
    graus_f = np.append(graus, 360.0)
    amp_f   = np.append(amp_por_fase, amp_por_fase[0])

    cor_principal = '#3498db' if not alerta_harmonicas else '#e74c3c'

    fig = plt.figure(figsize=(11, 4.5))

    # Painel esquerdo: polar
    ax_pol = fig.add_subplot(121, projection='polar')
    ax_pol.plot(theta_p, r_p, lw=2.0, color=cor_principal)
    ax_pol.fill(theta_p, r_p, alpha=0.25, color=cor_principal)
    ax_pol.axhline(nivel_uniforme, color='gray', lw=1.0, ls='--',
                   label=f'Uniforme (1/{n_bins_plt})')
    ax_pol.set_theta_zero_location('N')
    ax_pol.set_theta_direction(-1)
    ax_pol.set_rlabel_position(135)
    ax_pol.tick_params(labelsize=7)
    ax_pol.set_title(
        f'Polar\nMI = {mi:.5f}',
        fontsize=9, pad=14
    )

    # Painel direito: cartesiano
    ax_cart = fig.add_subplot(122)
    ax_cart.fill_between(graus_f, amp_f, nivel_uniforme,
                          where=(amp_f > nivel_uniforme),
                          alpha=0.30, color=cor_principal,
                          label='Amp > uniforme')
    ax_cart.fill_between(graus_f, amp_f, nivel_uniforme,
                          where=(amp_f <= nivel_uniforme),
                          alpha=0.15, color='gray',
                          label='Amp <= uniforme')
    ax_cart.plot(graus_f, amp_f, lw=2.0, color=cor_principal)
    ax_cart.axhline(nivel_uniforme, color='gray', lw=1.0, ls='--',
                    label=f'Uniforme (1/{n_bins_plt})')

    ax_cart.set_xlabel('Fase theta (graus)', fontsize=9)
    ax_cart.set_ylabel('Amplitude normalizada', fontsize=9)
    ax_cart.set_xlim(-180, 180)
    ax_cart.set_xticks([-180, -90, 0, 90, 180])
    ax_cart.tick_params(labelsize=8)
    ax_cart.legend(fontsize=7, loc='upper right', framealpha=0.7)
    ax_cart.grid(True, alpha=0.25)
    ax_cart.spines['top'].set_visible(False)
    ax_cart.spines['right'].set_visible(False)

    if alerta_harmonicas:
        ax_cart.text(0.02, 0.96,
                     'ATENCAO: Harmonica theta em Gamma Lento\n  (interpretar com cautela)',
                     transform=ax_cart.transAxes,
                     fontsize=7, color='#c0392b', va='top',
                     bbox=dict(boxstyle='round,pad=0.3', fc='#fde8e8',
                               ec='#c0392b', alpha=0.8))

    titulo = (
        f'PAC theta-gamma -- {stem}  |  {area_str}\n'
        f'Fase: {banda_fase[0]}-{banda_fase[1]} Hz  x  '
        f'Amp: {banda_amp[0]}-{banda_amp[1]} Hz   '
        f'MI = {mi:.5f}'
    )
    fig.suptitle(titulo, fontsize=9, y=1.02)
    plt.tight_layout()

    tag = f'{banda_amp[0]}-{banda_amp[1]}hz'
    caminho = pasta_saida / f'pac_{slug_area(area)}_{tag}.png'
    fig.savefig(caminho, dpi=130, bbox_inches='tight')
    plt.close(fig)
    return caminho


# Pares PAC a calcular
PARES_PAC = [
    {
        'banda_fase': (6, 12),
        'banda_amp':  (65, 110),
        'nome': 'theta-gamma_rapido',
        'hipotese': 'principal',
        'nota': 'Robusto -- fora do alcance das harmonicas de theta'
    },
    {
        'banda_fase': (6, 12),
        'banda_amp':  (25, 55),
        'nome': 'theta-gamma_lento',
        'hipotese': 'secundaria',
        'nota': 'Interpretar com cautela -- verificar flag de harmonicas (Fase 10C)'
    },
]

print('OK Fase 12 atualizada: plotar_pac() -- painel polar + cartesiano lado a lado.')
print('   Area anatomica no titulo | referencia uniforme visivel | alerta de harmonicas.')
print('   Hipotese PRINCIPAL  -> Theta (6-12 Hz) <-> Fast Gamma (65-110 Hz)')
print('   Hipotese SECUNDARIA -> Theta (6-12 Hz) <-> Slow Gamma (25-55 Hz)')
print('   Controle de harmonicas: flag automatico via Fase 10C')


OK Fase 12 atualizada: plotar_pac() -- painel polar + cartesiano lado a lado.
   Area anatomica no titulo | referencia uniforme visivel | alerta de harmonicas.
   Hipotese PRINCIPAL  -> Theta (6-12 Hz) <-> Fast Gamma (65-110 Hz)
   Hipotese SECUNDARIA -> Theta (6-12 Hz) <-> Slow Gamma (25-55 Hz)
   Controle de harmonicas: flag automatico via Fase 10C


---
## Fase 12 — Z-score de potências por banda por canal

**O que faz:**  
Normaliza a potência de cada banda por canal usando z-score calculado sobre toda a sessão (todos os segundos).  
Também gera z-score inter-sessão por rato (baseline = média de todas as sessões daquele rato).

**Por que:**  
O z-score remove variações absolutas de amplitude entre ratos e eletrodos, deixando apenas variações relativas dentro do contexto de cada animal — essencial para comparar sessões 25%/50%/75%.

In [20]:
def zscore_potencias(df_bandas, colunas_pot):
    """Calcula z-score de cada coluna de potência sobre toda a sessão.

    Parâmetros
    ----------
    df_bandas   : DataFrame com uma linha por segundo e colunas de potência
    colunas_pot : lista de colunas a normalizar (ex: ['pot_theta_ch00', ...])

    Retorna
    -------
    DataFrame com colunas _zscore adicionadas
    """
    df_z = df_bandas.copy()
    for col in colunas_pot:
        vals = df_z[col].values.astype(float)
        mu  = np.nanmean(vals)
        std = np.nanstd(vals)
        df_z[f'{col}_zscore'] = (vals - mu) / (std + 1e-12)
    return df_z


print('✅ Função zscore_potencias() definida.')

✅ Função zscore_potencias() definida.


---
## Fase 13 — Espectrograma (0–50 Hz)

**O que faz:**  
Calcula e plota espectrograma STFT limitado a 0–50 Hz, com janela de Hanning de 2s e overlap 50%.  
Salva PNG e array `.npy` para uso posterior em CNN.

**Por que 50 Hz?**  
O gamma lento vai até 55 Hz, mas o limite de 50 Hz mantém as bandas theta, harmônicas de theta e beta completamente visíveis sem incluir a região do notch (55–65 Hz).

In [21]:
import os
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import spectrogram
from pathlib import Path
from scipy.ndimage import gaussian_filter  # Aplica o blur para visual contínuo

SPEC_NPERSEG  = int(0.5 * FS)
SPEC_NOVERLAP = SPEC_NPERSEG // 2
SPEC_NFFT     = 512


def calcular_espectrograma(sig, fs=FS, f_max=None):
    if f_max is None:
        f_max = max(f2 for _, (_, f2) in BANDAS.items())

    freqs, t_spec, Sxx = spectrogram(
        sig,
        fs=fs,
        window='hann',
        nperseg=min(SPEC_NPERSEG, len(sig)),
        noverlap=min(SPEC_NOVERLAP, len(sig) // 2),
        nfft=SPEC_NFFT,
        scaling='density',
        mode='psd'
    )
    mask = freqs <= f_max
    return freqs[mask], t_spec, Sxx[mask, :]


def plotar_espectrogramas_por_banda(freqs, t_spec, Sxx, stem, area, pasta_saida,
                                     rato=None):
    """Espectrograma de Painel Único — Sinal Completo.

    Gera um gráfico único contínuo e suavizado do espectro total (0 - f_max Hz).
    Utiliza colormap 'jet' e estilo visual limpo (box off).

    `area` é o rótulo (string) da área anatômica (sinal já é a média dos
    canais bons daquela área).
    """
    pasta_base = Path(pasta_saida)
    pasta_base.mkdir(parents=True, exist_ok=True)

    Sxx_db_full = 10 * np.log10(Sxx + 1e-12)
    
    # Aplica o Blur (Filtro Gaussiano) para emular a interpolação suave do MATLAB
    Sxx_db_full_blur = gaussian_filter(Sxx_db_full, sigma=[0.8, 1.2])

    # Configuração de figura simples de painel único
    fig, ax = plt.subplots(figsize=(16, 5))
    
    vmin = np.percentile(Sxx_db_full_blur, 2)
    vmax = np.percentile(Sxx_db_full_blur, 99)

    # Plot único com o colormap 'jet' solicitado e sombreamento suavizado
    im = ax.pcolormesh(
        t_spec, freqs, Sxx_db_full_blur,
        shading='gouraud', cmap='jet',
        vmin=vmin, vmax=vmax
    )
    
    # Barra de cores lateral
    plt.colorbar(im, ax=ax, label='Potência (dB re uV²/Hz)', pad=0.01, fraction=0.02)

    # Configurações de eixos e limites reais
    ax.set_ylabel('Frequência (Hz)', fontsize=9)
    ax.set_xlabel('Tempo (s)', fontsize=9)
    ax.set_xlim(t_spec[0], t_spec[-1])
    ax.set_ylim(freqs[0], freqs[-1])
    
    # Título do painel contendo identificação e área anatômica
    ax.set_title(
        f'Espectrograma Completo — {stem}  |  {area}',
        fontsize=10, fontweight='bold'
    )
    
    # Formatação de bordas limpas (Estilo 'box off' do MATLAB)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.tick_params(labelsize=8)

    plt.tight_layout()

    # Salva a imagem do espectrograma completo
    caminho_png = pasta_base / f'spec_completo_{slug_area(area)}.png'
    fig.savefig(caminho_png, dpi=150, bbox_inches='tight')
    plt.close(fig)

    # Salva a matriz de dados em float32 para a rede neural convolucional (CNN)
    caminho_npy = pasta_base / f'espectrograma_{slug_area(area)}.npy'
    np.save(caminho_npy, Sxx.astype(np.float32))

    return [caminho_png], caminho_npy


print('OK Fase 14 atualizada: plotar_espectrogramas_por_banda() — Painel único do sinal completo.')
print('   Layout: gráfico contínuo total (0-f_max Hz), colormap jet com suavização gaussiana ativa.')
print('   Removidos GridSpec e loops internos por sub-bandas.')

OK Fase 14 atualizada: plotar_espectrogramas_por_banda() — Painel único do sinal completo.
   Layout: gráfico contínuo total (0-f_max Hz), colormap jet com suavização gaussiana ativa.
   Removidos GridSpec e loops internos por sub-bandas.


In [22]:
def plotar_figura1_lfp(sinal_por_canal, fs, canais_por_area, stem, pasta_saida,
                        rato=None, duracao_plot_s=1.0,
                        offset_inicio_s=0.0):
    from scipy.signal import welch, butter, sosfiltfilt

    if isinstance(sinal_por_canal, np.ndarray):
        sinal_dict = {ch: sinal_por_canal[ch] for ch in range(sinal_por_canal.shape[0])}
    else:
        sinal_dict = sinal_por_canal

    def _butter_bp(sig, fs, f1, f2, order=4):
        sos = butter(order, [f1, f2], btype='bandpass', fs=fs, output='sos')
        return sosfiltfilt(sos, sig)

    def _grupo(area):
        a = area.upper()
        if 'CA1' in a:  return 'CA1'
        if 'CA3' in a:  return 'CA3'
        if 'DGD' in a:  return 'DGD'
        if 'DGV' in a:  return 'DGV'
        return area

    # ── Paleta expandida por SUB-ÁREA ────────────────────────────────────────
    CORES_SUBAREA = {
        # CA1
        'CA1':             '#4472C4',
        # DGD (azul-roxo → roxo claro)
        'DGD hilos':       '#7030A0',
        'DGD granular':    '#9B59B6',
        'DGD molecular':   '#C39BD3',
        'DGD':             '#7030A0',   # fallback genérico
        # CA3 (vermelho → laranja)
        'CA3-c':           '#C00000',
        'CA3c':            '#C00000',
        'CA3b':            '#E05C00',
        'CA3':             '#C00000',   # fallback genérico
        # DGV (verde escuro → verde claro)
        'DGV hilos':       '#375623',
        'DGV granular':    '#538135',
        'DGV molecular':   '#70AD47',
        'DGV':             '#375623',   # fallback genérico
    }

    def _cor(area_str):
        # Tenta match exato primeiro (normalizado sem sufixo numérico)
        area_limpa = area_str.strip()
        import re
        area_limpa = re.sub(r'\s*\d+$', '', area_limpa)   # remove "...1" / "...2"
        if area_limpa in CORES_SUBAREA:
            return CORES_SUBAREA[area_limpa]
        # Fallback por grupo macro
        return CORES_SUBAREA.get(_grupo(area_limpa), '#555555')

    # ── CORREÇÃO PRINCIPAL: um canal representativo por SUB-ÁREA ─────────────
    # (antes era um por grupo macro → só 4 traços para CA1/CA3/DGD/DGV)
    subareas_vistas = set()
    canais_repr = []   # lista de (ch_idx, area_str, grupo)
    for area, chs in canais_por_area.items():
        import re
        area_limpa = re.sub(r'\s*\d+$', '', area.strip())   # normaliza "CA3b 1" → "CA3b"
        if area_limpa not in subareas_vistas and len(chs) > 0:
            canais_repr.append((chs[0], area_limpa, _grupo(area_limpa)))
            subareas_vistas.add(area_limpa)

    if not canais_repr:
        fallback_areas = ['CA1', 'DGD', 'CA3', 'DGV']
        for idx, g in enumerate(fallback_areas):
            if idx < len(sinal_dict):
                canais_repr.append((idx, g, g))

    # Janela de tempo
    n_amostras_total = len(next(iter(sinal_dict.values())))
    idx0  = int(offset_inicio_s * fs)
    idx1  = min(idx0 + int(duracao_plot_s * fs), n_amostras_total)
    t_seg = np.arange(idx1 - idx0) / fs

    # ── Layout ──────────────────────────────────────────────────────────────
    fig = plt.figure(figsize=(16, max(5, 1.4 * len(canais_repr) + 2)))
    gs_outer = gridspec.GridSpec(1, 3, figure=fig,
                                  width_ratios=[0.9, 2.2, 1.4], wspace=0.38)

    # ── Painel A: esquema anatômico (inalterado) ─────────────────────────────
    ax_anat = fig.add_subplot(gs_outer[0])
    ax_anat.set_xlim(0, 10); ax_anat.set_ylim(0, 10)
    ax_anat.set_aspect('equal'); ax_anat.axis('off')
    ax_anat.set_title('A', fontsize=12, fontweight='bold', loc='left', pad=8)
    hipp_bg = mpatches.FancyBboxPatch((1.0, 0.5), 8.0, 9.0,
        boxstyle='round,pad=0.3', linewidth=1.2, edgecolor='#555', facecolor='#f0ede8')
    ax_anat.add_patch(hipp_bg)
    camadas = [
        ('CA1',          (1.5, 8.2), '#4472C4', 0.9),
        ('DGD molecular',(1.5, 7.0), '#C39BD3', 0.7),
        ('DGD granular', (1.5, 5.9), '#9B59B6', 0.7),
        ('DGD hilos',    (1.5, 4.8), '#7030A0', 0.8),
        ('CA3b / CA3c',  (1.5, 3.5), '#C00000', 0.9),
        ('DGV hilos',    (1.5, 2.4), '#375623', 0.6),
        ('DGV granular', (1.5, 1.7), '#538135', 0.6),
        ('DGV molecular',(1.5, 1.0), '#70AD47', 0.6),
    ]
    for label, (x, y), cor, altura in camadas:
        ret = mpatches.FancyBboxPatch((x, y), 7.0, altura,
            boxstyle='round,pad=0.1', linewidth=0.8,
            edgecolor='#888', facecolor=cor, alpha=0.25)
        ax_anat.add_patch(ret)
        ax_anat.text(x + 3.5, y + altura / 2, label,
                     ha='center', va='center', fontsize=7, color='#222')
    pos_eletrodos_y = [8.65, 7.35, 6.25, 5.2, 4.0, 2.7, 2.0, 1.3]
    for y_el in pos_eletrodos_y:
        ax_anat.plot(5.0, y_el, 'o', color='#2c3e50', markersize=4, zorder=5)
    ax_anat.plot([5.0, 5.0], [pos_eletrodos_y[0], pos_eletrodos_y[-1]],
                 '-', color='#2c3e50', lw=1.5, zorder=4)
    ax_anat.text(5.0, 9.7, 'Array\n(16 ch)', ha='center', fontsize=7.5,
                 color='#2c3e50', fontweight='bold')

    # ── Painel B: LFP theta — um traço por SUB-ÁREA ──────────────────────────
    ax_lfp = fig.add_subplot(gs_outer[1])
    ax_lfp.set_title('B  --  LFP (theta 6-12 Hz)', fontsize=10,
                      fontweight='bold', loc='left')

    n_ch_plot = len(canais_repr)
    offset_step = 1.0
    yticks_pos, yticks_labels = [], []
    max_amp = 0.0
    traces = []
    for ch_idx, area_str, grupo in canais_repr:
        sig_raw = sinal_dict.get(ch_idx, np.zeros(n_amostras_total))
        seg = sig_raw[idx0:idx1].astype(float)
        seg_th = _butter_bp(seg, fs, 6, 12)
        std = seg_th.std() + 1e-12
        seg_norm = seg_th / std
        max_amp = max(max_amp, np.abs(seg_norm).max())
        traces.append((seg_norm, area_str, grupo, ch_idx))

    offset_step = max(offset_step, max_amp * 2.2)

    for i, (seg_norm, area_str, grupo, ch_idx) in enumerate(traces):
        offset_y = (n_ch_plot - 1 - i) * offset_step
        cor = _cor(area_str)
        ax_lfp.plot(t_seg, seg_norm + offset_y, lw=0.9, color=cor, alpha=0.9)
        yticks_pos.append(offset_y)
        yticks_labels.append(f'ch{ch_idx+1:02d}\n({area_str})')

    ax_lfp.set_yticks(yticks_pos)
    ax_lfp.set_yticklabels(yticks_labels, fontsize=7)
    ax_lfp.set_xlabel('Tempo (s)', fontsize=9)
    ax_lfp.set_xlim(t_seg[0], t_seg[-1])
    y_barra = -offset_step * 0.6
    ax_lfp.plot([t_seg[-1] - 0.2, t_seg[-1]], [y_barra, y_barra], 'k-', lw=2)
    ax_lfp.text(t_seg[-1] - 0.1, y_barra - offset_step * 0.15,
                '200 ms', ha='center', fontsize=7)
    ax_lfp.spines['top'].set_visible(False)
    ax_lfp.spines['right'].set_visible(False)
    ax_lfp.spines['bottom'].set_visible(False)
    ax_lfp.tick_params(bottom=False, labelbottom=False)

    grupos_vistos = {g for _, _, g in canais_repr}
    CORES_AREA_MACRO = {'CA1': '#4472C4', 'CA3': '#C00000',
                        'DGD': '#7030A0', 'DGV': '#375623'}
    legend_els = [Line2D([0], [0], color=c, lw=2, label=g)
                  for g, c in CORES_AREA_MACRO.items() if g in grupos_vistos]
    ax_lfp.legend(handles=legend_els, fontsize=7, loc='upper right', framealpha=0.7)

    # ── Painel C: PSD Welch ───────────────────────────────────────────────────
    ax_psd = fig.add_subplot(gs_outer[2])
    ax_psd.set_title('C  --  PSD Welch', fontsize=10, fontweight='bold', loc='left')

    for ch_idx, area_str, grupo in canais_repr:
        sig_raw = sinal_dict.get(ch_idx, np.zeros(n_amostras_total))
        nperseg = min(fs, len(sig_raw))
        freqs_w, psd_w = welch(sig_raw.astype(float), fs=fs,
                                window='hann', nperseg=nperseg,
                                noverlap=nperseg // 2, nfft=2048, scaling='density')
        cor = _cor(area_str)
        mask_plot = freqs_w <= 120
        ax_psd.semilogy(freqs_w[mask_plot], psd_w[mask_plot],
                         lw=1.0, color=cor, alpha=0.85,
                         label=f'ch{ch_idx+1:02d} ({area_str})')

    ax_psd.axvspan(6,  12,  alpha=0.12, color='#3498db', label='Theta (6-12 Hz)')
    ax_psd.axvspan(25, 55,  alpha=0.08, color='#2ecc71', label='Gamma lento')
    ax_psd.axvspan(65, 110, alpha=0.08, color='#e74c3c', label='Gamma rápido')
    ax_psd.axvspan(55, 65,  alpha=0.06, color='red',     label='Notch 55-65 Hz')
    ax_psd.set_xlabel('Frequência (Hz)', fontsize=9)
    ax_psd.set_ylabel('PSD (uV²/Hz)', fontsize=9)
    ax_psd.set_xlim(0, 120)
    ax_psd.tick_params(labelsize=7)
    ax_psd.legend(fontsize=6, loc='upper right', framealpha=0.7)
    ax_psd.grid(True, alpha=0.2)
    ax_psd.spines['top'].set_visible(False)
    ax_psd.spines['right'].set_visible(False)

    rato_str = f'Rato {rato}  |  ' if rato else ''
    fig.suptitle(
        f'Figura 1 -- Histologia e registros eletrofisiológicos\n{rato_str}{stem}',
        fontsize=11, y=1.01, fontweight='bold')
    plt.tight_layout()

    pasta_saida = Path(pasta_saida)
    pasta_saida.mkdir(parents=True, exist_ok=True)
    caminho = pasta_saida / 'figura1_histologia_lfp.png'
    fig.savefig(caminho, dpi=180, bbox_inches='tight')
    plt.close(fig)
    return caminho

---
## Pipeline Principal — Execução por arquivo

Executa todas as Fases 1–15 em sequência para cada arquivo `.int`.  
Os resultados são salvos em `EDA_v6_outputs/<stem>/`.

**Seletor de rato:** altere `RATO_PIPELINE` antes de rodar:
```python
RATO_PIPELINE = 'R1'   # processa só o R1
RATO_PIPELINE = 'ALL'  # processa todos os ratos
```

**Ordem de execução interna por arquivo:**
1. Leitura do `.int`
2. Identificação de canais ruins
3. Filtro notch + remoção DC
4. Downsample → 1 kHz
5. Plot sinal bruto + **Figura 1** (histologia + LFP theta + PSD)
6. Por canal bom: pico theta, harmônicas, wavelet, espectrograma, PSD
7. Por segundo × canal: stats, Welch (bandas + centroide espectral), PAC (plot no s=0)
8. Z-score de potências
9. Salva `features.parquet` + relatório


In [23]:
from tqdm.auto import tqdm
import gc
import numpy as np
import pandas as pd

# ══════════════════════════════════════════════════════════════════════════════
# SELETOR DE RATO
# Defina RATO_PIPELINE para processar um rato específico ou 'ALL' para todos.
# Exemplos:
#   RATO_PIPELINE = 'R1'
#   RATO_PIPELINE = 'R3'
#   RATO_PIPELINE = 'ALL'
# ══════════════════════════════════════════════════════════════════════════════
RATO_PIPELINE = 'ALL'   # ← altere aqui antes de rodar

# ── Filtragem do dataframe de arquivos ───────────────────────────────────────
_rato_upper = RATO_PIPELINE.strip().upper()
if _rato_upper == 'ALL':
    df_run = df_arquivos.copy()
    print(f'▶ Modo ALL: {len(df_run)} arquivos de {df_run["rato"].nunique()} ratos.')
else:
    df_run = df_arquivos[df_arquivos['rato'].str.upper() == _rato_upper].copy()
    if df_run.empty:
        ratos_disponiveis = sorted(df_arquivos['rato'].dropna().unique())
        raise ValueError(
            f"Rato '{RATO_PIPELINE}' não encontrado em df_arquivos.\n"
            f"Ratos disponíveis: {ratos_disponiveis}"
        )
    print(f'▶ Modo rato único: {_rato_upper} — {len(df_run)} arquivos.')

print(df_run.groupby(['rato', 'condicao', 'trial']).size()
      .rename('n_arquivos').reset_index().to_string(index=False))
print()

# ══════════════════════════════════════════════════════════════════════════════
# PIPELINE PRINCIPAL
# ══════════════════════════════════════════════════════════════════════════════
relatorio = []

for idx, row in tqdm(df_run.iterrows(), total=len(df_run),
                     desc=f'Processando [{_rato_upper}]'):

    stem = row['stem']
    path = row['path']
    rato = row['rato']
    anot = row['acontencimento']

    # ── Estrutura de pastas: OUTPUT_DIR / <rato> / <stem> / <subpasta_por_saida> ──
    pasta = OUTPUT_DIR / rato / stem
    subpastas = {
        'info':          pasta / '00_info',
        'sinal_bruto':   pasta / '01_sinal_bruto',
        'downsample':    pasta / '02_downsample',
        'por_segundo':   pasta / '03_por_segundo',
        'psd_welch':     pasta / '04_psd_welch',
        'harmonicas':    pasta / '05_harmonicas',
        'espectrograma': pasta / '06_espectrograma',
        'pac':           pasta / '07_pac',
        'features':      pasta / '08_features',
    }
    for _sub in subpastas.values():
        _sub.mkdir(parents=True, exist_ok=True)

    try:
        # ── 1. LEITURA ─────────────────────────────
        t_orig, canais_ativos, data_neural, _ = read_intan_data(str(path))
        data_neural = np.asarray(data_neural, dtype=np.float32)

        # Recoloca cada canal ativo na sua posição física real (canal N → linha N-1).
        # Canais não ativos na gravação ficam com zero, e serão pegos como
        # "canal morto" pelo identificar_canais_ruins (RMS/std ~ 0).
        amps = np.zeros((N_CANAIS, data_neural.shape[1]), dtype=np.float32)
        for pos, canal_fisico in enumerate(canais_ativos):
            if 1 <= canal_fisico <= N_CANAIS:
                amps[canal_fisico - 1] = data_neural[pos]

        if len(canais_ativos) < N_CANAIS:
            print(f'ℹ️  {stem}: {len(canais_ativos)}/{N_CANAIS} canais ativos '
                f'{sorted(canais_ativos.tolist())} — demais tratados como canal morto.')

        # ── 2. CANAIS RUINS ────────────────────────
        canais_info = identificar_canais_ruins(amps, rato, stem, anot)

        canais_bons = [ch for ch, info in canais_info.items() if not info['ruim']]

        if not canais_bons:
            pd.DataFrame([
                {'canal': ch+1, **info} for ch, info in canais_info.items()
            ]).to_csv(subpastas['info'] / 'canais_info.csv', index=False)
            relatorio.append({
                'stem': stem, 'status': 'DESCARTADO',
                'motivo': 'Todos os canais identificados como ruins'
            })
            print(f'⚠️  {stem}: todos os canais ruins — pulando.')
            continue

        pd.DataFrame([
            {'canal': ch+1, **info} for ch, info in canais_info.items()
        ]).to_csv(subpastas['info'] / 'canais_info.csv', index=False)

        # ── 3. FILTRAGEM ───────────────────────────
        amps = aplicar_notch(amps, FS_ORIGINAL)
        amps = remover_dc_offset(amps)

        # ── 4. DOWNSAMPLE ──────────────────────────
        data_down = downsample(amps, FS_ORIGINAL, FS)

        del amps
        gc.collect()

        t_down = np.arange(data_down.shape[1]) / FS

        salvar_downsample(data_down, stem, subpastas['downsample'], metadados={
            'rato': rato,
            'condicao': row['condicao'],
            'trial': row['trial'],
            'acontencimento': anot,
            'canais_bons': canais_bons,
            'canais_ativos_hardware': sorted(canais_ativos.tolist()),  # novo
        })

        sinal_por_segundo(data_down, stem, subpastas['por_segundo'], canais_bons=canais_bons)

        # ── 5. PLOTS GLOBAIS DA SESSÃO ─────────────
        plotar_sinal_bruto(data_down, t_down, stem, canais_info, subpastas['sinal_bruto'])

        # ── 5.5 SINAL MÉDIO POR ÁREA + JANELAMENTO DO WELCH (Fase 10D) ──
        # A partir daqui a análise é por ÁREA anatômica (média dos canais bons
        # daquela área; se só houver 1 canal bom na área, usa-se ele sozinho).
        sinais_area, areas_canais_usados = sinal_medio_por_area(data_down, rato, canais_bons)
        areas = sorted(sinais_area.keys())

        if not areas:
            relatorio.append({
                'stem': stem, 'status': 'DESCARTADO',
                'motivo': 'Nenhuma área anatômica mapeada para os canais bons (confira CANAL_AREAS/planilha)'
            })
            print(f'⚠️  {stem}: sem área anatômica mapeada — pulando.')
            continue

        salvar_janelamento_welch(sinais_area, stem, pasta_saida_base=WELCH_JANELAMENTO_DIR / rato)

        # ── 6. FEATURES + PLOTS POR ÁREA ───────────
        n_segundos = data_down.shape[1] // FS
        features_sessao = []

        theta_peaks = {}
        harmonicas_por_area = {}
        psds_por_area = {}

        for area in areas:
            sig_full = sinais_area[area]

            theta_peak_hz, theta_peak_psd, theta_mean_psd, freqs, psd = detectar_pico_theta(sig_full)

            harmonicas = calcular_harmonicas(theta_peak_hz, n_harmonicas=4)
            alerta, detalhes = verificar_alerta_pac(harmonicas, freqs, psd)

            theta_peaks[area] = theta_peak_hz
            harmonicas_por_area[area] = {
                'theta_peak_hz': theta_peak_hz,
                'harmonicas': harmonicas,
                'alerta': alerta,
                'freqs': freqs,
                'psd': psd
            }
            psds_por_area[area] = (freqs, psd)

            plotar_espectro_harmonicas(
                freqs, psd, theta_peak_hz, harmonicas, alerta,
                stem, area, subpastas['harmonicas']
            )

            freqs_spec, t_spec, Sxx = calcular_espectrograma(sig_full, fs=FS)
            plotar_espectrogramas_por_banda(freqs_spec, t_spec, Sxx, stem, area,
                                            subpastas['espectrograma'], rato=rato)
            del Sxx
            gc.collect()

        if psds_por_area:
            freqs_ref = harmonicas_por_area[areas[0]]['freqs']
            plotar_psd(freqs_ref, psds_por_area, areas, stem, subpastas['psd_welch'], rato=rato)
            plotar_psd_por_canal(freqs_ref, psds_por_area, areas, stem, subpastas['psd_welch'], rato=rato)

        # ── 7. LOOP POR SEGUNDO ────────────────────
        for s in range(n_segundos):

            ini = s * FS
            fim = ini + FS

            row_feat = {
                'stem': stem,
                'rato': rato,
                'condicao': row['condicao'],
                'trial': row['trial'],
                'segundo': s
            }

            for area in areas:

                sig = sinais_area[area][ini:fim]
                prefix = slug_area(area)

                stats = features_estatisticas(sig)
                for k, v in stats.items():
                    row_feat[f'{prefix}_{k}'] = v

                freqs_w, psd_w, pots = psd_welch_bandas(sig, fs=FS)
                for k, v in pots.items():
                    row_feat[f'{prefix}_{k}'] = v

                # Fase 10B — frequência contínua (centroide espectral)
                row_feat[f'{prefix}_centroide_hz'] = centroide_espectral(freqs_w, psd_w)

                for par in PARES_PAC:
                    try:
                        mi, amp_por_fase, bins_centro = calcular_pac(
                            sig, fs=FS,
                            banda_fase=par['banda_fase'],
                            banda_amp=par['banda_amp']
                        )
                        row_feat[f'{prefix}_pac_{par["nome"]}'] = mi

                        if par['nome'] == 'theta-gamma_lento':
                            row_feat[f'{prefix}_pac_sl_alerta'] = \
                                harmonicas_por_area[area]['alerta']

                        # Plot PAC apenas no primeiro segundo (evita excesso de arquivos)
                        if s == 0:
                            plotar_pac(
                                amp_por_fase, bins_centro, mi,
                                stem, area,
                                banda_fase=par['banda_fase'],
                                banda_amp=par['banda_amp'],
                                pasta_saida=subpastas['pac'],
                                rato=rato,
                                alerta_harmonicas=harmonicas_por_area[area]['alerta']
                            )

                    except Exception:
                        row_feat[f'{prefix}_pac_{par["nome"]}'] = np.nan

            features_sessao.append(row_feat)

        df_feat = pd.DataFrame(features_sessao)

        df_feat.to_parquet(subpastas['features'] / 'features.parquet', index=False)

        del df_feat
        gc.collect()

        # ── 9. RELATÓRIO ───────────────────────────
        relatorio.append({
            'stem': stem,
            'rato': rato,
            'condicao': row['condicao'],
            'trial': row['trial'],
            'status': 'OK',
            'n_segundos': n_segundos
        })

        del data_down, sinais_area, theta_peaks, harmonicas_por_area, psds_por_area
        gc.collect()

    except Exception as e:
        relatorio.append({'stem': stem, 'status': 'ERRO', 'motivo': str(e)})
        print(f'❌ ERRO em {stem}: {e}')
        gc.collect()

# ── FINAL ──────────────────────────────────────
df_relatorio = pd.DataFrame(relatorio)
df_relatorio.to_csv(OUTPUT_DIR / f'relatorio_{_rato_upper}.csv', index=False)

print('\n' + '═' * 60)
print(f'RELATÓRIO FINAL  [{_rato_upper}]')
print(df_relatorio.groupby('status').size().to_string())

▶ Modo ALL: 103 arquivos de 7 ratos.
rato condicao trial  n_arquivos
  R1     0.25    T1           1
  R1     0.25    T2           1
  R1      0.5    T1           1
  R1      0.5    T2           1
  R1      0.5    T3           1
  R1      0.5    T4           1
  R1     0.75    T1           1
  R1     0.75    T2           1
  R1     0.75    T3           1
  R1     0.75    T4           1
  R2     0.25    T1           1
  R2     0.25    T2           1
  R2     0.25    T3           1
  R2     0.25    T4           1
  R2      0.5    T1           1
  R2      0.5    T2           1
  R2      0.5    T3           1
  R2      0.5    T4           1
  R2     0.75    T1           1
  R2     0.75    T2           1
  R2     0.75    T3           1
  R2     0.75    T4           1
  R2      NOR    T1           2
  R2      NOR    T2           1
  R2      NOR    T3           1
  R2      NOR    T4           1
  R3     0.25    T1           1
  R3     0.25    T2           1
  R3     0.25    T3           1
  R

Processando [ALL]:   0%|          | 0/103 [00:00<?, ?it/s]


Data file contains 213.78 seconds of data from 16 amplifier channels.
Channels: 1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16


Processando [ALL]:   1%|          | 1/103 [00:07<13:24,  7.89s/it]

⚠️  r1_25_T1_240416_124853: todos os canais ruins — pulando.

Data file contains 252.45 seconds of data from 16 amplifier channels.
Channels: 1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16


...


Data file contains 233.19 seconds of data from 14 amplifier channels.
Channels: 1 2 3 4 5 6 7 9 10 11 12 14 15 16
ℹ️  R8NORT3_241108_124108: 14/16 canais ativos [1, 2, 3, 4, 5, 6, 7, 9, 10, 11, 12, 14, 15, 16] — demais tratados como canal morto.


Processando [ALL]:  99%|█████████▉| 102/103 [1:08:04<00:47, 47.10s/it]


Data file contains 228.09 seconds of data from 14 amplifier channels.
Channels: 1 2 3 4 5 6 7 9 10 11 12 14 15 16
ℹ️  R8NORT4_241108_125739: 14/16 canais ativos [1, 2, 3, 4, 5, 6, 7, 9, 10, 11, 12, 14, 15, 16] — demais tratados como canal morto.


Processando [ALL]: 100%|██████████| 103/103 [1:08:51<00:00, 40.11s/it]


════════════════════════════════════════════════════════════
RELATÓRIO FINAL  [ALL]
status
DESCARTADO     7
OK            96


In [24]:
# ══════════════════════════════════════════════════════════════
# Consolidação — features_all.parquet / features_all.csv
# Junta todos os features.parquet gerados pelo pipeline
# em um único DataFrame e salva na raiz de OUTPUT_DIR.
# ══════════════════════════════════════════════════════════════
import pandas as pd
import json
from pathlib import Path

dfs = []
# Estrutura: OUTPUT_DIR / <rato> / <stem> / 08_features / features.parquet
for pasta_rato in sorted(OUTPUT_DIR.iterdir()):
    if not pasta_rato.is_dir():
        continue
    for pasta in sorted(pasta_rato.iterdir()):
        if not pasta.is_dir():
            continue
        fp = pasta / '08_features' / 'features.parquet'
        if not fp.exists():
            continue
        df = pd.read_parquet(fp)
        if df.empty:
            continue

        # Garante que rato / condicao / trial estejam presentes
        # (prioridade: coluna no df > json de metadados na pasta 02_downsample > nomes das pastas)
        if 'rato' not in df.columns or 'condicao' not in df.columns:
            jsons = list((pasta / '02_downsample').glob('*.json'))
            if jsons:
                meta = json.loads(jsons[0].read_text())
                if 'rato' not in df.columns:
                    df.insert(0, 'rato', meta.get('rato', pasta_rato.name))
                if 'condicao' not in df.columns:
                    df.insert(1, 'condicao', str(meta.get('condicao', '')))
                if 'trial' not in df.columns:
                    df.insert(2, 'trial', meta.get('trial', ''))
                if 'stem' not in df.columns:
                    df.insert(3, 'stem', pasta.name)
            else:
                partes = pasta.name.split('_')
                if 'rato' not in df.columns:
                    df.insert(0, 'rato', pasta_rato.name)
                if 'condicao' not in df.columns:
                    df.insert(1, 'condicao', partes[1] if len(partes) > 1 else '')
                if 'trial' not in df.columns:
                    df.insert(2, 'trial', partes[2] if len(partes) > 2 else '')
                if 'stem' not in df.columns:
                    df.insert(3, 'stem', pasta.name)

        dfs.append(df)

if not dfs:
    print('⚠️  Nenhum features.parquet encontrado em OUTPUT_DIR.')
else:
    features_all = pd.concat(dfs, ignore_index=True)

    if 'rato' in features_all.columns:
        features_all = features_all[features_all['rato'] != RATO_EXCLUIR].copy()

    out_parquet = OUTPUT_DIR / 'features_all.parquet'
    out_csv     = OUTPUT_DIR / 'features_all.csv'

    features_all.to_parquet(out_parquet, index=False)
    features_all.to_csv(out_csv, index=False)

    print(f'✅ features_all salvo em {OUTPUT_DIR}')
    print(f'   Linhas : {len(features_all):,}')
    print(f'   Colunas: {len(features_all.columns):,}')
    print(f'   Ratos  : {sorted(features_all["rato"].unique()) if "rato" in features_all.columns else "—"}')
    print(f'   Arquivo parquet : {out_parquet.stat().st_size / 1024:.1f} KB')
    print(f'   Arquivo CSV     : {out_csv.stat().st_size / 1024:.1f} KB')
    display(features_all.head(3))

✅ features_all salvo em EDA_outputs
   Linhas : 19,285
   Colunas: 85
   Ratos  : ['R1', 'R2', 'R3', 'R4', 'R5', 'R7', 'R8']
   Arquivo parquet : 12237.4 KB
   Arquivo CSV     : 25388.7 KB


,stem,rato,condicao,trial,segundo,CA3c_mean,CA3c_std,CA3c_rms,CA3c_kurtosis,CA3c_skewness,...,CA3b_pot_gamma_lento,CA3b_pot_gamma_rapido,CA3b_theta_gamma_lento_ratio,CA3b_theta_gamma_rapido_ratio,CA3b_delta_theta_ratio,CA3b_entropia_espectral,CA3b_centroide_hz,CA3b_pac_theta-gamma_rapido,CA3b_pac_theta-gamma_lento,CA3b_pac_sl_alerta
0,r1_50_T1_240418_113900,R1,0.5,T1,0,-7.978905,23.773988,25.077189,-0.235833,-0.162295,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,r1_50_T1_240418_113900,R1,0.5,T1,1,-1.368893,20.476143,20.521850,3.437346,-0.628033,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,r1_50_T1_240418_113900,R1,0.5,T1,2,2.805976,61.944318,62.007838,-0.071365,0.214923,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


---
## Pós-Pipeline — Análise e Diagnóstico

As células abaixo **não fazem parte do pipeline principal** e devem ser rodadas  
**após** a célula Pipeline ter sido executada ao menos uma vez.

> **Sobre o rato processado:** nenhuma célula desta seção pede o rato de novo.  
> **Z-score comparativo** e **Consolidação** sempre varrem todos os ratos já
> presentes em `OUTPUT_DIR`. **PAC** reaproveita automaticamente o valor de
> `RATO_PIPELINE` definido na célula do Pipeline Principal (o mesmo rato — ou
> `'ALL'` — que você processou por último).

| Célula | O que faz |
|--------|-----------|
| **Tendência** | Remove colunas `_zscore`/`_tendencia` antigas de `features_all` e adiciona colunas `_tendencia` (slope do z-score por trial); ignora sessões com `nor` no nome |
| **Z-score comparativo** | Plota z-score de potência por banda × seção (25%/50%/75%) para cada rato |
| **PAC** | Recalcula e plota PAC sobre o sinal completo, para o(s) rato(s) definido(s) em `RATO_PIPELINE` |
| **Sumário de outputs** | Descreve a estrutura de arquivos gerados |


---
## Tendência (slope) do z-score por trial

**O que faz:**  
Para cada trial (`stem`), calcula o z-score de cada coluna de potência usando como baseline **apenas aquele trial** e, em seguida, ajusta uma regressão linear simples do z-score em função do tempo (`segundo`). A inclinação (slope) dessa reta vira a feature `<coluna>_tendencia`: **1 valor por trial** (positivo = potência cresce ao longo da sessão, negativo = cai).

**Por que trocar `_zscore` por `_tendencia`:**  
Colunas `_zscore` por segundo têm média ~0 dentro de cada trial — pouco úteis como feature de nível para modelos que comparam trials entre si. A tendência captura a dinâmica (sobe/desce) de forma mais informativa e compacta.

**Sessões `NOR`:** trials cujo `stem` contém `nor` (case-insensitive) **não são processados** aqui — ficam com `NaN` nas colunas `_tendencia`, sem alterar o restante de `features_all`.

> Requer que a célula **Consolidação** já tenha gerado `features_all.parquet` em `OUTPUT_DIR`.

In [25]:
# ══════════════════════════════════════════════════════════════
# Tendência (slope) do z-score de potência por trial ao longo do tempo.
# Substitui as colunas _zscore (por segundo, pouco úteis como feature de
# nível — a média de cada trial é ~0) por colunas _tendencia (1 valor por
# trial, capturando se a potência sobe/desce ao longo da sessão).
# Trials com 'nor' no nome (stem) NÃO são processados.
# ══════════════════════════════════════════════════════════════
import re
import warnings
import numpy as np
import pandas as pd

PATH_FEATURES_ALL = OUTPUT_DIR / 'features_all.parquet'
PATH_FEATURES_CSV = PATH_FEATURES_ALL.with_suffix('.csv')


def zscore_potencias_trial(df_bandas, colunas_pot):
    """Z-score de cada coluna de potência, usando como baseline as linhas
    recebidas (aqui: só as linhas de um único trial — captura a dinâmica
    interna da sessão, diferente do zscore_potencias() da Fase 12, cujo
    baseline é o rato inteiro).

    Colunas 100% NaN no trial (área não gravada naquela sessão) permanecem
    NaN, sem gerar RuntimeWarning.
    """
    df_z = df_bandas.copy()
    for col in colunas_pot:
        vals = df_z[col].values.astype(float)
        if np.isnan(vals).all():
            df_z[f'{col}_zscore'] = np.nan
            continue
        mu = np.nanmean(vals)
        std = np.nanstd(vals)
        df_z[f'{col}_zscore'] = (vals - mu) / (std + 1e-12)
    return df_z


def slope_tendencia(t, y):
    """Inclinação de uma regressão linear simples de y em função do tempo t."""
    mask = ~np.isnan(y) & ~np.isnan(t)
    if mask.sum() < 2:
        return np.nan
    coef = np.polyfit(t[mask], y[mask], deg=1)
    return coef[0]


df = pd.read_parquet(PATH_FEATURES_ALL)

# ── Remove _zscore e _tendencia de rodadas anteriores ANTES do merge ────
# (evita colunas _tendencia_x/_tendencia_y duplicadas se esta célula já
# tiver rodado antes sobre o mesmo parquet, e garante que _zscore nunca
# fica salvo em features_all — só _tendencia é persistida.)
cols_antigas = [c for c in df.columns
                if c.endswith('_zscore') or re.search(r'_tendencia(_x|_y)?$', c)]
if cols_antigas:
    print(f'Removendo {len(cols_antigas)} coluna(s) _zscore/_tendencia de rodada(s) anterior(es).')
    df = df.drop(columns=cols_antigas)

# ── Exclui trials NOR do processamento ───────────────────────────────────
mask_nor = df['stem'].str.contains('nor', case=False, na=False)
stems_nor = sorted(df.loc[mask_nor, 'stem'].unique())
if stems_nor:
    print(f"⏭️  Ignorando {len(stems_nor)} trial(s) com 'nor' no nome (não processados):")
    for s in stems_nor:
        print(f'    {s}')
df_proc = df[~mask_nor].copy()

colunas_pot = [c for c in df_proc.columns
               if re.search(r'_pot_(delta|theta|beta|gamma_lento|gamma_rapido)$', c)]
print(f'\n{len(colunas_pot)} colunas de potência bruta usadas para gerar tendência.')

registros_tendencia = []
trials_com_area_ausente = []

for stem, g in df_proc.groupby('stem'):
    with warnings.catch_warnings():
        warnings.simplefilter('ignore', category=RuntimeWarning)
        g_z = zscore_potencias_trial(g, colunas_pot)   # baseline = só este trial

    t = g_z['segundo'].values.astype(float)
    row = {'stem': stem}
    n_areas_ausentes_neste_trial = 0
    for col in colunas_pot:
        if g_z[col].isna().all():
            n_areas_ausentes_neste_trial += 1
        row[f'{col}_tendencia'] = slope_tendencia(t, g_z[f'{col}_zscore'].values)
    if n_areas_ausentes_neste_trial:
        trials_com_area_ausente.append((stem, n_areas_ausentes_neste_trial))
    registros_tendencia.append(row)

df_tendencia = pd.DataFrame(registros_tendencia)
print(f'{df_tendencia.shape[1] - 1} feature(s) de tendência geradas para '
      f'{len(df_tendencia)} trial(s) (excluindo NOR).')

if trials_com_area_ausente:
    print(f'\nℹ️  {len(trials_com_area_ausente)} trial(s) com pelo menos 1 área ausente '
          f'(gerando NaN esperado em algumas colunas de tendência):')
    for stem, n in sorted(trials_com_area_ausente, key=lambda x: -x[1])[:10]:
        print(f'    {stem}: {n} área(s) ausente(s)')

# Merge de volta em TODO o df (linhas NOR ficam com _tendencia = NaN,
# já que não foram processadas — não são removidas de features_all)
df = df.merge(df_tendencia, on='stem', how='left', validate='many_to_one')

# ── Checagem de sanidade: garante que o merge não duplicou colunas ──────
cols_duplicadas = [c for c in df.columns if c.endswith(('_tendencia_x', '_tendencia_y'))]
assert not cols_duplicadas, (
    f'Merge gerou colunas duplicadas inesperadas: {cols_duplicadas[:5]}... '
    f'Verifique se PATH_FEATURES_ALL realmente foi limpo antes do merge.'
)

df.to_parquet(PATH_FEATURES_ALL, index=False)
df.to_csv(PATH_FEATURES_CSV, index=False)

print('\n✅ features_all atualizado: colunas _tendencia adicionadas (sem _zscore).')
print(f'Shape final: {df.shape}')


⏭️  Ignorando 22 trial(s) com 'nor' no nome (não processados):
    R2NORT1_240428_125837
    R2NORT1_240428_125837 (1)
    R2NORT2_240428_131343
    R2NORT3_240428_132842
    R2NORT4_240428_134346
    R3NORT2_240916_121730
    R3NORT3_240916_123200
    R4NORT1_240926_121735
    R4NORT2_240926_123150
    R4NORT3_240926_124557
    R4NORT4_240926_130032
    R5NORT1_241003_120058
    R5NORT2_241003_121631
    R5NORT3_241003_123213
    R5NORT4_241003_124723
    R7NORT1_241014_122657
    R7NORT2_241014_124518
    R7NORT3_241014_130441
    R8NORT1_241108_121043
    R8NORT2_241108_122600
    R8NORT3_241108_124108
    R8NORT4_241108_125739

20 colunas de potência bruta usadas para gerar tendência.
20 feature(s) de tendência geradas para 72 trial(s) (excluindo NOR).

ℹ️  13 trial(s) com pelo menos 1 área ausente (gerando NaN esperado em algumas colunas de tendência):
    r8_75_T3_241112_131327: 15 área(s) ausente(s)
    r3_75_T3_240925_124820: 10 área(s) ausente(s)
    r1_50_T1_240418_113900: 5 

In [31]:
import re
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from pathlib import Path

# ══════════════════════════════════════════════════════════════
# CONFIGURAÇÃO
# ══════════════════════════════════════════════════════════════
SECOES            = [0.25, 0.50, 0.75]
BANDAS_PLOT       = list(BANDAS.keys())
CORES_SECAO       = {0.25: '#3498db', 0.50: '#e67e22', 0.75: '#e74c3c'}
CORES_BANDAS_PLOT = CORES_BANDAS

PASTA_ZSCORE = OUTPUT_DIR / 'Z-score'
PASTA_ZSCORE.mkdir(exist_ok=True)

# ── Z-score comparativo por banda x seção (CORRIGIDO: baseline = rato inteiro) ──
registros = []
for pasta_rato in sorted(OUTPUT_DIR.iterdir()):
    if not pasta_rato.is_dir() or pasta_rato.name == 'Z-score':
        continue

    # ── junta TODOS os trials do rato antes de normalizar ──
    dfs_rato = []
    for pasta in sorted(pasta_rato.iterdir()):
        fp = pasta / '08_features' / 'features.parquet'
        if fp.exists():
            df_t = pd.read_parquet(fp)
            if not df_t.empty:
                dfs_rato.append(df_t)
    if not dfs_rato:
        continue

    df_rato = pd.concat(dfs_rato, ignore_index=True)

    # ── z-score calculado sobre o rato inteiro (baseline correto p/ comparar sessões) ──
    cols_pot = [c for c in df_rato.columns if re.search(r'_pot_(delta|theta|beta|gamma_lento|gamma_rapido)$', c)]
    df_rato = zscore_potencias(df_rato, cols_pot)

    rato = pasta_rato.name
    for banda in BANDAS_PLOT:
        cols_z = [c for c in df_rato.columns if f'_pot_{banda}_zscore' in c]
        for col in cols_z:
            area_label = col.split('_pot_')[0]
            for cond, g in df_rato.groupby('condicao'):
                registros.append({
                    'rato': rato, 'condicao': str(cond),
                    'area': area_label, 'banda': banda,
                    'zscore': g[col].mean(),
                })

df_plot = pd.DataFrame(registros)
df_plot['condicao_num'] = pd.to_numeric(df_plot['condicao'], errors='coerce')
df_plot = df_plot[df_plot['condicao_num'].isin(SECOES)].copy()
df_plot = df_plot[df_plot['rato'] != RATO_EXCLUIR].copy()

# ══════════════════════════════════════════════════════════════
# PLOTAGEM — grade área × banda por rato
# ══════════════════════════════════════════════════════════════
if df_plot.empty:
    print('⚠️  Nenhum dado encontrado.')
else:
    ratos      = sorted(df_plot['rato'].unique())
    secoes_arr = np.array(SECOES)
    n_bandas   = len(BANDAS_PLOT)

    for rato in ratos:
        df_r  = df_plot[df_plot['rato'] == rato]
        areas = sorted(df_r['area'].unique())
        n_area = len(areas)

        if n_area == 0:
            continue

        # grade: áreas (linhas) × bandas (colunas)
        fig, axes = plt.subplots(
            n_area, n_bandas,
            figsize=(n_bandas * 3.5, n_area * 2.8),
            sharex=True
        )
        if n_area == 1: axes = axes[np.newaxis, :]
        if n_bandas == 1: axes = axes[:, np.newaxis]

        fig.suptitle(
            f'Z-score de potência por banda × seção — {rato} — todas as áreas\n'
            f'Seta verde → cresce 25%→75% | vermelha → cai | cinza → estável',
            fontsize=11
        )

        for c_idx, area in enumerate(areas):
            df_rc = df_r[df_r['area'] == area]

            for b_idx, banda in enumerate(BANDAS_PLOT):
                ax = axes[c_idx, b_idx]

                df_rb = (df_rc[df_rc['banda'] == banda]
                           .groupby('condicao_num')['zscore'].mean()
                           .reindex(SECOES))
                vals = df_rb.values

                mask_valido = ~np.isnan(vals)
                if mask_valido.sum() >= 2:
                    ax.plot(secoes_arr[mask_valido], vals[mask_valido],
                            color='black', lw=1.5, zorder=2)

                for sec, val in zip(SECOES, vals):
                    if not np.isnan(val):
                        ax.scatter(sec, val, color=CORES_SECAO[sec], s=60, zorder=3)

                v25, v75 = vals[0], vals[2]
                if not (np.isnan(v25) or np.isnan(v75)):
                    delta    = v75 - v25
                    cor_seta = '#2ecc71' if delta > 0.05 else \
                               '#e74c3c' if delta < -0.05 else '#aaaaaa'
                    ax.annotate('', xy=(0.75, v75), xytext=(0.25, v25),
                                arrowprops=dict(arrowstyle='->', color=cor_seta, lw=1.8))

                ax.axhline(0, color='#cccccc', lw=0.8, ls='--')
                ax.yaxis.set_major_formatter(ticker.FormatStrFormatter('%.2f'))
                ax.tick_params(labelsize=6)
                ax.set_xlim(0.20, 0.80)
                ax.xaxis.set_major_locator(ticker.FixedLocator(SECOES))
                ax.xaxis.set_major_formatter(ticker.FixedFormatter(
                    [f'{int(s*100)}%' for s in SECOES]))

                if c_idx == 0:
                    ax.set_title(banda.replace('_', ' ').upper(), fontsize=9,
                                 color=CORES_BANDAS_PLOT.get(banda, 'black'))
                if b_idx == 0:
                    ax.set_ylabel(area.upper(), fontsize=8, rotation=0,
                                  ha='right', va='center')

        fig.text(0.5, 0.01, 'Seção (%)', ha='center', fontsize=9)
        plt.tight_layout(rect=[0, 0.03, 1, 0.95])

        caminho = PASTA_ZSCORE / f'zscore_{rato}.png'
        fig.savefig(caminho, dpi=150, bbox_inches='tight')
        plt.close(fig)
        print(f'✅ {rato}: salvo em {caminho}')

    print(f'\nTotal: {len(ratos)} plots gerados em {PASTA_ZSCORE}')

KeyboardInterrupt: 

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import json

# ══════════════════════════════════════════════════════════════
# Reaproveita o MESMO seletor da célula Pipeline (RATO_PIPELINE) —
# não é preciso indicar o rato de novo aqui.
# Se RATO_PIPELINE = 'ALL', recalcula PAC para todos os ratos já
# processados; se for um rato específico (ex: 'R3'), recalcula só esse.
# ══════════════════════════════════════════════════════════════
if 'RATO_PIPELINE' not in dir():
    raise NameError(
        "RATO_PIPELINE não definido — rode a célula do Pipeline Principal "
        "pelo menos uma vez antes desta (ela define qual rato usar aqui)."
    )

_rato_pac = RATO_PIPELINE.strip().upper()

if _rato_pac == 'ALL':
    pastas_rato = sorted(p for p in OUTPUT_DIR.iterdir() if p.is_dir() and p.name != 'Z-score')
else:
    pastas_rato = [OUTPUT_DIR / _rato_pac]

print(f'▶ PAC — reaproveitando RATO_PIPELINE = {_rato_pac!r} ({len(pastas_rato)} rato(s))')

resultados_pac = []
n_ok = 0
n_pulados = 0

for pasta_rato in pastas_rato:
    rato = pasta_rato.name

    if not pasta_rato.exists():
        print(f'⏭️  {rato}: pasta não existe — pulando.')
        n_pulados += 1
        continue

    stems_disponiveis = sorted(p.name for p in pasta_rato.iterdir() if p.is_dir())

    for stem in stems_disponiveis:
        pasta = pasta_rato / stem
        fp_sinal = pasta / '02_downsample' / f'{stem}_sinal_downsample.npy'
        fp_info  = pasta / '00_info' / 'canais_info.csv'

        if not fp_sinal.exists() or not fp_info.exists():
            print(f'⏭️  {rato}/{stem}: arquivos não encontrados — pulando.')
            n_pulados += 1
            continue

        try:
            data_down = np.load(fp_sinal)
            df_ch       = pd.read_csv(fp_info)
            canais_bons = df_ch[df_ch['ruim'] == False]['canal'].sub(1).tolist()

            sinais_area, areas_canais_usados = sinal_medio_por_area(data_down, rato, canais_bons)
            areas = sorted(sinais_area.keys())

            if not areas:
                print(f'⏭️  {rato}/{stem}: sem área anatômica mapeada — pulando.')
                n_pulados += 1
                continue

            pasta_pac = pasta / '07_pac'

            for area in areas:
                sig = sinais_area[area]

                for par in PARES_PAC:
                    try:
                        mi, amp_por_fase, bins_centro = calcular_pac(
                            sig, fs=FS,
                            banda_fase=par['banda_fase'],
                            banda_amp=par['banda_amp']
                        )

                        plotar_pac(
                            amp_por_fase, bins_centro, mi,
                            stem=stem, area=area,
                            banda_fase=par['banda_fase'],
                            banda_amp=par['banda_amp'],
                            pasta_saida=pasta_pac,
                            rato=rato
                        )

                        resultados_pac.append({
                            'rato':     rato,
                            'stem':     stem,
                            'area':     area,
                            'canais_usados': areas_canais_usados[area],
                            'par':      par['nome'],
                            'hipotese': par['hipotese'],
                            'mi':       round(mi, 6),
                        })

                    except Exception as e:
                        print(f'  ❌ {rato}/{stem} | {area} | {par["nome"]}: {e}')

            print(f'✅ {rato}/{stem}: {len(areas)} área(s) processada(s).')
            n_ok += 1

        except Exception as e:
            print(f'⏭️  {rato}/{stem}: erro inesperado ({e}) — pulando.')
            n_pulados += 1
            continue

df_pac = pd.DataFrame(resultados_pac)
print(f'\n✅ {n_ok} arquivo(s) processado(s) | ⏭️ {n_pulados} pulado(s) | {len(df_pac)} linhas de PAC no total')
display(df_pac)


▶ PAC — reaproveitando RATO_PIPELINE = 'ALL' (7 rato(s))
⏭️  R1/r1_25_T1_240416_124853: arquivos não encontrados — pulando.
⏭️  R1/r1_25_T2_240416_130351: arquivos não encontrados — pulando.
  ❌ R1/r1_50_T1_240418_113900 | DGD | theta-gamma_rapido: 'Text' object has no attribute 'eventson'
✅ R1/r1_50_T1_240418_113900: 3 área(s) processada(s).
✅ R1/r1_50_T2_240418_115409: 4 área(s) processada(s).


In [8]:

arquivo = OUTPUT_DIR / 'features_all.csv'

df = pd.read_csv(arquivo, low_memory=False)

df["condicao"] = (
    pd.to_numeric(df["condicao"], errors="coerce")
      .round(2)
)

areas = [
    "CA3b",
    "CA3c",
    "DGD",
    "DGV",
]

condicoes = [0.25, 0.50, 0.75]
trials = ["T1", "T2", "T3", "T4"]

linhas = []

for rato in sorted(df["rato"].unique()):

    for cond in condicoes:

        sessao = int(cond * 100)

        mensagens = []

        for trial in trials:

            sub = df[
                (df["rato"] == rato) &
                (df["condicao"] == cond) &
                (df["trial"] == trial)
            ]

            if sub.empty:
                mensagens.append(f"{trial}: SEM ARQUIVO")
                continue

            faltando = []

            for area in areas:

                cols = [
                    c for c in df.columns
                    if c.startswith(area + "_")
                    and not c.endswith("_tendencia")
                ]

                if len(cols) == 0:
                    continue

                if sub[cols].isna().all().all():
                    faltando.append(area)

            if faltando:
                mensagens.append(f"{trial}: {', '.join(faltando)}")

        # resumo da sessão
        if len(mensagens) == 0:
            linhas.append(f"{rato} - Sessao {sessao}% - falta areas: nenhuma")

        elif len(mensagens) == 4 and all("SEM ARQUIVO" in m for m in mensagens):
            linhas.append(
                f"{rato} - Sessao {sessao}% - SEM ARQUIVO "
                f"({' | '.join(mensagens)})"
            )

        else:
            linhas.append(
                f"{rato} - Sessao {sessao}% - "
                + " | ".join(mensagens)
            )

texto = "\n".join(linhas)

print(texto)

caminho_relatorio = OUTPUT_DIR / 'relatorio_areas_faltantes.txt'
caminho_relatorio.write_text(texto, encoding="utf-8")

print(f"\n✅ Relatório salvo em {caminho_relatorio}")

R1 - Sessao 25% - SEM ARQUIVO (T1: SEM ARQUIVO | T2: SEM ARQUIVO | T3: SEM ARQUIVO | T4: SEM ARQUIVO)
R1 - Sessao 50% - T1: CA3b | T3: CA3b
R1 - Sessao 75% - T1: SEM ARQUIVO | T2: SEM ARQUIVO | T3: SEM ARQUIVO
R2 - Sessao 25% - falta areas: nenhuma
R2 - Sessao 50% - falta areas: nenhuma
R2 - Sessao 75% - falta areas: nenhuma
R3 - Sessao 25% - T1: CA3b | T2: CA3b | T3: CA3b | T4: CA3b
R3 - Sessao 50% - falta areas: nenhuma
R3 - Sessao 75% - T3: CA3c, DGV | T4: CA3c
R4 - Sessao 25% - falta areas: nenhuma
R4 - Sessao 50% - falta areas: nenhuma
R4 - Sessao 75% - T4: SEM ARQUIVO
R5 - Sessao 25% - falta areas: nenhuma
R5 - Sessao 50% - falta areas: nenhuma
R5 - Sessao 75% - falta areas: nenhuma
R7 - Sessao 25% - T3: CA3b
R7 - Sessao 50% - T2: SEM ARQUIVO | T3: SEM ARQUIVO | T4: SEM ARQUIVO
R7 - Sessao 75% - T1: CA3b | T2: CA3b
R8 - Sessao 25% - falta areas: nenhuma
R8 - Sessao 50% - SEM ARQUIVO (T1: SEM ARQUIVO | T2: SEM ARQUIVO | T3: SEM ARQUIVO | T4: SEM ARQUIVO)
R8 - Sessao 75% - T3: CA3c

---
## Sumário dos Outputs

Após a execução completa, a estrutura de pastas será organizada por **rato → arquivo →
tipo de saída**:

```
EDA_10_outputs/
├── relatorio_<RATO>.csv                     ← status de cada arquivo processado
├── features_all.parquet                     ← dataset central (todas as áreas/arquivos/ratos)
├── features_all.csv
└── <RATO>/
    └── <stem>/
        ├── 00_info/
        │   └── canais_info.csv              ← RMS/std/kurtosis + motivo de exclusão por canal
        ├── 01_sinal_bruto/
        │   └── sinal_bruto.png              ← traçado completo (canais ruins em vermelho)
        ├── 02_downsample/
        │   ├── <stem>_sinal_downsample.npy  ← sinal limpo [N_CANAIS × N_amostras @ 1kHz]
        │   └── <stem>_sinal_downsample.json ← metadados (rato, condição, trial, canais_bons)
        ├── 03_por_segundo/
        │   └── <stem>_por_segundo.parquet   ← sinal fatiado por segundo (por canal)
        ├── 04_psd_welch/
        │   ├── psd_welch.png                ← PSD Welch — todas as áreas sobrepostas
        │   └── psd_welch_por_canal.png       ← PSD Welch — um subplot por área
        ├── 05_harmonicas/
        │   └── harmonicas_theta_<area>.png  ← espectro anotado com pico θ + harmônicas
        ├── 06_espectrograma/
        │   ├── spec_completo_<area>.png     ← espectrograma (STFT) por área
        │   └── espectrograma_<area>.npy     ← matriz do espectrograma (para uso em CNN)
        ├── 07_pac/
        │   ├── pac_<area>_65-110hz.png      ← PAC theta → gamma rápido (HIPÓTESE PRINCIPAL)
        │   └── pac_<area>_25-55hz.png       ← PAC theta → gamma lento (com flag de harmônicas)
        └── 08_features/
            └── features.parquet             ← stats + Welch + centroide + PAC, por área/segundo

welch_janelamento/
└── <RATO>/
    └── <stem>/
        ├── <area>_janelas_sinal.npy         ← segmentos brutos no tempo usados no Welch (1s, 50% overlap)
        ├── <area>_janelas_psd.npy           ← PSD de cada janela individual (antes da média final)
        └── <area>_janelas_meta.json         ← frequências, tempo de início de cada janela, parâmetros
```

> **Unidade de análise = área anatômica**, não canal. A área de cada canal vem de
> `planilha_ratos_R1_R7.xlsx` (lida dinamicamente — Fase 0). Quando uma área tem mais
> de 1 canal bom, o sinal usado nas Fases 10A→14 é a média ponto-a-ponto (domínio do
> tempo) desses canais.

### Bandas espectrais de interesse fisiológico

| Banda | Hz | Notas |
|-------|----|-------|
| Theta | 6–12 | Ritmo hipocampal principal |
| Beta | 13–30 | Processamento cognitivo |
| Gamma lento | 25–55 | PAC secundário — verificar flag de harmônicas |
| Gamma rápido | 65–110 | PAC principal — robusto contra harmônicas |

### Controle metodológico de harmônicas

| Coluna em `features_all` | Significado |
|--------------------------|-------------|
| `<area>_theta_peak_hz` (via `harmonicas_por_area`) | Frequência do pico θ real daquela área/registro |
| `<area>_pac_sl_alerta` | `True` se h3 ou h4 de θ caem em Slow Gamma (±3 Hz) |
| `<area>_centroide_hz` | Frequência contínua (Fase 10B) — centroide espectral por segundo/área |

> Registros com `alerta = True` não são excluídos.
> O PAC Theta–Slow Gamma desses registros deve ser interpretado com cautela adicional na análise estatística.
